# Docking Lab · Kaggle training and evidence

This notebook trains expert-initialized PPO policies on two independent T4 GPUs, evaluates the actual backend, and exports verifiable artifacts. It contains the project source so it works without a GitHub checkout. All distances are docking-port distances in metres. Capture is gated at **at most 1.25 m**, low relative speed, aligned attitude, and low rotation rate.

The learned policy supplies advisory context to the LangChain planner; the backend enforces constraints. Training reward alone is not a docking accuracy measurement. The final held-out evaluation uses fresh seeds, reports failures, and includes confidence intervals. Provider LLM calls are evaluated separately from deterministic-planner experiments.

Enable **GPU T4 ×2** and Internet in Kaggle settings. The two GPUs have separate memory; each seeded run uses one device. Stable-Baselines3's small MLP can be faster on CPU; GPU usage is provided for the requested runtime, not claimed as a speed advantage.

Recorded outputs below are attached from the actual completed Kaggle CLI runs in `proofs/kaggle-final-seed-*`. The training cells in this notebook document were not executed as a single Run All. Rerunning these cells starts new reproducible experiments under the notebook-labelled run directories.


In [1]:
from pathlib import Path
import json, os, sys, subprocess, time, hashlib
import torch

ROOT = Path('/kaggle/working/llm-docking') if Path('/kaggle/working').exists() else Path.cwd()
print('Python:', sys.version.split()[0], 'Torch:', torch.__version__)
print('CUDA devices:', torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(index, props.name, f'{props.total_memory / 2**30:.2f} GiB')
assert torch.cuda.is_available(), 'Select a GPU accelerator in Kaggle settings.'


Verified live Kaggle runtime: Python 3.12.13; CUDA devices: 2
0 Tesla T4 14.56 GiB
1 Tesla T4 14.56 GiB


## Restore the project

The next cell writes the Python source packaged with this notebook to ROOT. It creates no credentials and does not include the runtime URL or your `.env`. Existing model artifacts are preserved. Rerun only when you want the notebook version of the source to replace ROOT/src.


In [ ]:
PROJECT_FILES = json.loads('{"src/app/config.py": "\\"\\"\\"Server settings; provider keys stay in .env and never travel to the browser.\\"\\"\\"\\n\\nfrom functools import lru_cache\\n\\nfrom pydantic import Field\\nfrom pydantic_settings import BaseSettings, SettingsConfigDict\\n\\n\\nclass Settings(BaseSettings):\\n    model_config = SettingsConfigDict(env_prefix=\\"LLM_DOCKING_\\", env_file=\\".env\\", extra=\\"ignore\\")\\n    mode: str = \\"heuristic\\"\\n    llm_model: str = \\"gpt-4o\\"\\n    llm_model_groq: str = \\"llama-3.1-8b-instant\\"\\n    llm_model_openrouter: str = \\"meta-llama/llama-3.1-8b-instant\\"\\n    llm_base_url: str | None = None\\n    llm_timeout_s: float = Field(default=20.0, gt=0.0, le=120.0)\\n    llm_max_retries: int = Field(default=1, ge=0, le=3)\\n    llm_retry_backoff_s: float = Field(default=0.25, ge=0.0, le=5.0)\\n    rl_model_path: str | None = None\\n    rl_normalize_path: str | None = None\\n    event_buffer_size: int = 1000\\n    sim_default_controller: str = \\"lqr\\"\\n\\n\\n@lru_cache\\ndef get_settings() -> Settings:\\n    return Settings()\\n", "src/app/main.py": "\\"\\"\\"FastAPI application entrypoint.\\n\\nRun locally with::\\n\\n    uvicorn app.main:app --reload\\n\\nor via Docker (see Dockerfile / docker-compose.yml).\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport logging\\nimport sys\\nimport time\\nimport uuid\\nfrom contextlib import asynccontextmanager\\nfrom pathlib import Path\\n\\nimport structlog\\nfrom fastapi import FastAPI, Request\\nfrom fastapi.responses import JSONResponse\\nfrom fastapi.staticfiles import StaticFiles\\n\\nfrom app import __version__\\nfrom app.api.routes import router\\nfrom app.config import get_settings\\n\\n# ---------------------------------------------------------------------------\\n# Logging: stdlib plumbing + structlog rendering\\n# ---------------------------------------------------------------------------\\n\\n\\ndef _configure_logging() -> None:\\n    logging.basicConfig(\\n        format=\\"%(message)s\\",\\n        stream=sys.stdout,\\n        level=logging.INFO,\\n        force=True,\\n    )\\n    structlog.configure(\\n        processors=[\\n            structlog.contextvars.merge_contextvars,\\n            structlog.processors.add_log_level,\\n            structlog.processors.TimeStamper(fmt=\\"iso\\", utc=True),\\n            structlog.processors.StackInfoRenderer(),\\n            structlog.processors.format_exc_info,\\n            structlog.dev.ConsoleRenderer(),\\n        ],\\n        wrapper_class=structlog.make_filtering_bound_logger(logging.INFO),\\n        cache_logger_on_first_use=True,\\n    )\\n\\n\\n_configure_logging()\\nlog = structlog.get_logger(\\"llm_docking.main\\")\\n\\nsettings = get_settings()\\n\\n\\n@asynccontextmanager\\nasync def lifespan(_: FastAPI):\\n    log.info(\\n        \\"startup\\",\\n        mode=settings.mode,\\n        model=settings.llm_model if settings.mode != \\"heuristic\\" else \\"heuristic (no network)\\",\\n        controller=settings.sim_default_controller,\\n    )\\n    yield\\n    log.info(\\"shutdown\\")\\n\\n\\napp = FastAPI(\\n    title=\\"LLM Docking API\\",\\n    description=(\\n        \\"Backend for an LLM-augmented autonomous satellite rendezvous-and-docking \\"\\n        \\"agent: the reasoning layer re-plans structured sub-goals, low-level \\"\\n        \\"PID/LQR controllers track them, and every step streams over SSE.\\"\\n    ),\\n    version=__version__,\\n    lifespan=lifespan,\\n)\\n\\napp.include_router(router)\\n\\n\\n@app.middleware(\\"http\\")\\nasync def request_context_middleware(request: Request, call_next):\\n    request_id = uuid.uuid4().hex[:12]\\n    structlog.contextvars.bind_contextvars(request_id=request_id)\\n    start = time.perf_counter()\\n    response = await call_next(request)\\n    duration_ms = (time.perf_counter() - start) * 1000.0\\n    structlog.contextvars.clear_contextvars()\\n    log.info(\\n        \\"request\\",\\n        method=request.method,\\n        path=request.url.path,\\n        status=response.status_code,\\n        duration_ms=round(duration_ms, 2),\\n    )\\n    return response\\n\\n\\n@app.get(\\"/api\\", include_in_schema=False)\\nasync def root() -> JSONResponse:\\n    return JSONResponse(\\n        {\\n            \\"service\\": \\"llm-docking\\",\\n            \\"version\\": __version__,\\n            \\"mode\\": settings.mode,\\n            \\"docs\\": \\"/docs\\",\\n            \\"health\\": \\"/api/v1/health\\",\\n            \\"simulate\\": \\"POST /api/v1/docking/simulate\\",\\n        }\\n    )\\n\\n\\nproject_root = Path(__file__).resolve().parents[2]\\nif not (project_root / \\"frontend\\").exists():\\n    project_root = Path.cwd()\\nfrontend = project_root / \\"frontend\\" / \\"dist\\"\\nproofs = project_root / \\"proofs\\"\\nif proofs.is_dir():\\n    app.mount(\\"/proofs/files\\", StaticFiles(directory=proofs), name=\\"proof_files\\")\\n\\n\\n@app.get(\\"/proofs/\\", include_in_schema=False)\\nasync def proof_index():\\n    from fastapi.responses import HTMLResponse\\n\\n    files = [\\n        p\\n        for p in proofs.rglob(\\"*\\")\\n        if p.is_file() and p.suffix in {\\".json\\", \\".csv\\", \\".png\\", \\".md\\", \\".txt\\"}\\n    ]\\n    from html import escape\\n\\n    links = \\"\\".join(\\n        f\'<li><a href=\\"/proofs/files/{p.relative_to(proofs).as_posix()}\\">{escape(p.relative_to(proofs).as_posix())}</a></li>\'\\n        for p in files\\n    )\\n    return HTMLResponse(\\n        \'<html lang=\\"en\\"><meta name=\\"viewport\\" content=\\"width=device-width, initial-scale=1\\"><title>Docking evidence</title>\'\\n        \'<body style=\\"font-family:system-ui;max-width:900px;margin:40px auto;padding:20px\\"><a href=\\"/\\">Dashboard</a>\'\\n        \\"<h1>Training & evaluation evidence</h1><p>Measured artifacts, parameters, and hashes from actual runs.</p><ul>\\"\\n        + links\\n        + \\"</ul></body></html>\\"\\n    )\\n\\n\\nif frontend.is_dir():\\n    app.mount(\\"/\\", StaticFiles(directory=frontend, html=True), name=\\"dashboard\\")\\nelse:\\n    app.add_api_route(\\"/\\", root, include_in_schema=False)\\n", "src/app/__init__.py": "\\"\\"\\"LLM-augmented autonomous satellite rendezvous-and-docking simulation backend.\\"\\"\\"\\n\\n__version__ = \\"0.1.0\\"\\n", "src/app/api/routes.py": "\\"\\"\\"API routes for the docking simulator.\\n\\nSession lifecycle\\n-----------------\\n``POST /api/v1/docking/simulate`` validates the request, creates a session,\\nand starts the simulation as a background task; it returns 202 immediately\\nwith the session id and the stream/summary URLs. The simulation then pushes\\nevents into the session\'s queue, which the SSE endpoint streams to the client\\nuntil the terminal ``docked``/``failed`` event is emitted.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport asyncio\\nimport logging\\nimport os\\nfrom collections.abc import AsyncGenerator\\n\\nfrom fastapi import APIRouter, Header, HTTPException, Request\\nfrom fastapi.responses import JSONResponse\\nfrom sse_starlette.sse import EventSourceResponse\\n\\nfrom app.config import get_settings\\nfrom app.schemas.docking import ControllerType, SessionSummary, SimulateRequest, SimulateResponse\\nfrom app.sessions.store import InMemorySessionStore, Session\\nfrom app.simulation.runner import SimulationRunner\\n\\nlogger = logging.getLogger(\\"llm_docking.api\\")\\n\\nrouter = APIRouter(prefix=\\"/api/v1\\", tags=[\\"docking\\"])\\n\\n# Single in-memory store for the process. Swappable for a Redis-backed store\\n# behind the same SessionStore interface.\\nstore = InMemorySessionStore(queue_size=get_settings().event_buffer_size)\\n\\n# How long the SSE endpoint waits for the next event before re-checking\\n# whether the session has finished (keeps the loop responsive to termination).\\n_SSE_POLL_TIMEOUT_S = 0.5\\n\\n\\n@router.post(\\n    \\"/docking/simulate\\",\\n    status_code=202,\\n    response_model=SimulateResponse,\\n    summary=\\"Start a docking simulation\\",\\n)\\nasync def start_simulation(request: Request, body: SimulateRequest) -> SimulateResponse:\\n    if sum(not s.finished.is_set() for s in store._sessions.values()) >= 4:\\n        raise HTTPException(\\n            status_code=429, detail=\\"Four sessions are active; stop one before starting another\\"\\n        )\\n    _validate_controller_ready(body)\\n    from app.llm.reasoning_agent import provider_has_key\\n\\n    provider = body.config.provider or get_settings().mode\\n    if not provider_has_key(provider):\\n        raise HTTPException(\\n            status_code=400, detail=f\\"Configure a server-side API key for {provider}\\"\\n        )\\n    session = await store.create()\\n    session.stream_pacing_s = body.config.stream_pacing_s\\n    try:\\n        runner = SimulationRunner(session=session, request=body)\\n    except (ValueError, ImportError) as exc:\\n        await store.close(session.session_id)\\n        raise HTTPException(status_code=400, detail=str(exc)) from None\\n    session.task = asyncio.create_task(runner.run())\\n    logger.info(\\"simulation_started\\", extra={\\"session_id\\": session.session_id})\\n    return SimulateResponse(\\n        session_id=session.session_id,\\n        stream_url=f\\"/api/v1/docking/sessions/{session.session_id}/stream\\",\\n        summary_url=f\\"/api/v1/docking/sessions/{session.session_id}\\",\\n    )\\n\\n\\n@router.get(\\n    \\"/docking/sessions/{session_id}/stream\\",\\n    summary=\\"SSE stream of simulation events\\",\\n)\\nasync def stream_events(\\n    session_id: str, after: int = 0, last_event_id: str | None = Header(default=None)\\n) -> EventSourceResponse:\\n    session = await _get_session(session_id)\\n    try:\\n        cursor = int(last_event_id) if last_event_id else after\\n    except ValueError:\\n        raise HTTPException(status_code=400, detail=\\"Last-Event-ID must be an integer\\") from None\\n    if cursor < 0 or cursor > len(session.event_log):\\n        raise HTTPException(status_code=400, detail=\\"Event cursor is outside this session\\")\\n\\n    async def event_generator() -> AsyncGenerator[dict, None]:\\n        index = cursor\\n        while True:\\n            while index < len(session.event_log):\\n                event = session.event_log[index]\\n                index += 1\\n                yield {\\n                    \\"id\\": str(event.event_id),\\n                    \\"event\\": event.type,\\n                    \\"data\\": event.model_dump_json(),\\n                }\\n            if session.finished.is_set():\\n                break\\n            await asyncio.sleep(0.05)\\n        logger.info(\\"stream_closed\\", extra={\\"session_id\\": session_id})\\n\\n    return EventSourceResponse(event_generator())\\n\\n\\n@router.get(\\n    \\"/docking/sessions/{session_id}\\",\\n    response_model=SessionSummary,\\n    summary=\\"Poll the session summary\\",\\n)\\nasync def get_summary(session_id: str) -> SessionSummary:\\n    session = await _get_session(session_id)\\n    if session.summary is None:\\n        raise HTTPException(status_code=404, detail=\\"Session has no summary yet\\")\\n    return session.summary\\n\\n\\n@router.get(\\n    \\"/docking/sessions/{session_id}/events\\",\\n    summary=\\"Dump the full ordered event log\\",\\n)\\nasync def get_events(session_id: str) -> list[dict]:\\n    session = await _get_session(session_id)\\n    return [e.model_dump(mode=\\"json\\") for e in session.event_log]\\n\\n\\n@router.get(\\"/health\\", summary=\\"Liveness probe\\")\\nasync def health() -> JSONResponse:\\n    return JSONResponse({\\"status\\": \\"ok\\"})\\n\\n\\n@router.get(\\"/capabilities\\")\\nasync def capabilities():\\n    from app.llm.reasoning_agent import provider_has_key\\n\\n    settings = get_settings()\\n    return {\\n        \\"providers\\": {\\n            p: provider_has_key(p)\\n            for p in [\\"heuristic\\", \\"openai\\", \\"openrouter\\", \\"groq\\", \\"anthropic\\"]\\n        },\\n        \\"default_provider\\": settings.mode,\\n        \\"rl_available\\": bool(\\n            settings.rl_model_path\\n            and settings.rl_normalize_path\\n            and os.path.isfile(settings.rl_model_path)\\n            and os.path.isfile(settings.rl_normalize_path)\\n        ),\\n        \\"capture_range_m\\": 1.25,\\n    }\\n\\n\\n@router.post(\\"/docking/sessions/{session_id}/cancel\\")\\nasync def cancel_session(session_id: str):\\n    session = await _get_session(session_id)\\n    if session.finished.is_set():\\n        raise HTTPException(status_code=409, detail=\\"Session has already finished\\")\\n    session.cancel_requested = True\\n    return {\\"status\\": \\"cancellation_requested\\"}\\n\\n\\n@router.post(\\"/docking/sessions/{session_id}/dock\\")\\nasync def request_docking(session_id: str):\\n    session = await _get_session(session_id)\\n    if session.finished.is_set() or not session.capture.get(\\"eligible\\"):\\n        raise HTTPException(\\n            status_code=409,\\n            detail={\\"message\\": \\"Capture gate is not satisfied\\", \\"capture\\": session.capture},\\n        )\\n    session.dock_requested = True\\n    return {\\"status\\": \\"capture_requested\\", \\"capture\\": session.capture}\\n\\n\\n@router.get(\\"/docking/sessions/{session_id}/export\\")\\nasync def export_session(session_id: str):\\n    session = await _get_session(session_id)\\n    return JSONResponse(\\n        {\\n            \\"schema_version\\": \\"1.0\\",\\n            \\"summary\\": session.summary.model_dump(mode=\\"json\\") if session.summary else None,\\n            \\"request\\": getattr(session, \\"request_data\\", None),\\n            \\"events\\": [e.model_dump(mode=\\"json\\") for e in session.event_log],\\n        },\\n        headers={\\"Content-Disposition\\": f\'attachment; filename=\\"docking-{session_id}.json\\"\'},\\n    )\\n\\n\\nasync def _get_session(session_id: str) -> Session:\\n    session = await store.get(session_id)\\n    if session is None:\\n        raise HTTPException(status_code=404, detail=f\\"Unknown session: {session_id}\\")\\n    return session\\n\\n\\ndef _validate_controller_ready(body: SimulateRequest) -> None:\\n    \\"\\"\\"Fail fast (400) when the requested run needs RL artifacts it hasn\'t got.\\n\\n    Both ``controller=\\"rl\\"`` and ``rl_advisor_enabled=True`` require the\\n    trained policy + VecNormalize stats (process-wide settings).\\n    \\"\\"\\"\\n    needs_rl = body.config.controller is ControllerType.RL or body.config.rl_advisor_enabled\\n    if not needs_rl:\\n        return\\n    settings = get_settings()\\n    missing = [\\n        name\\n        for name, path in (\\n            (\\"LLM_DOCKING_RL_MODEL_PATH\\", settings.rl_model_path),\\n            (\\"LLM_DOCKING_RL_NORMALIZE_PATH\\", settings.rl_normalize_path),\\n        )\\n        if not path or not os.path.isfile(path)\\n    ]\\n    if missing:\\n        reason = (\\n            \\"controller=\'rl\'\\"\\n            if body.config.controller is ControllerType.RL\\n            else \\"rl_advisor_enabled=True\\"\\n        )\\n        raise HTTPException(\\n            status_code=400,\\n            detail=(\\n                f\\"{reason} requires a trained policy: set \\"\\n                + \\" and \\".join(missing)\\n                + \\" (train with `python -m app.simulation.rl.train_ppo`)\\"\\n            ),\\n        )\\n", "src/app/api/__init__.py": "\\"\\"\\"HTTP API layer: session lifecycle, SSE streaming, summary polling.\\"\\"\\"\\n", "src/app/llm/reasoning_agent.py": "\\"\\"\\"A deterministic planner and a bounded LangChain tool-calling planner.\\n\\nThe LLM explains its chosen action and alternatives. Physics and capture checks\\nare enforced separately, and every failed provider call is reported as fallback.\\n\\"\\"\\"\\n\\nimport asyncio\\nimport json\\nimport os\\nfrom abc import ABC, abstractmethod\\n\\nimport numpy as np\\nfrom pydantic import BaseModel, Field\\n\\nfrom app.config import get_settings\\nfrom app.schemas.docking import DockingPhase\\nfrom app.schemas.events import AnomalyDetail, LlmSubgoal, RlAdvisorSuggestion\\n\\nPROVIDER_BASE_URLS = {\\n    \\"openai\\": \\"https://api.openai.com/v1\\",\\n    \\"openrouter\\": \\"https://openrouter.ai/api/v1\\",\\n    \\"groq\\": \\"https://api.groq.com/openai/v1\\",\\n}\\nPROVIDER_ENV_KEYS = {\\n    \\"openai\\": (\\"OPENAI_API_KEY\\",),\\n    \\"openrouter\\": (\\"OPENROUTER_API_KEY\\",),\\n    \\"groq\\": (\\"GROQ_API_KEY\\",),\\n    \\"anthropic\\": (\\"ANTHROPIC_API_KEY\\",),\\n}\\n\\n\\nclass ReasoningInput(BaseModel):\\n    step: int\\n    sim_time_s: float\\n    phase: str\\n    range_m: float\\n    position_m: list[float]\\n    velocity_mps: list[float]\\n    orientation_quat: list[float]\\n    angular_velocity_radps: list[float]\\n    fuel_remaining_pct: float\\n    controller: str\\n    anomaly_context: list[AnomalyDetail] = Field(default_factory=list)\\n    priority_change: str | None = None\\n    rl_advisor: RlAdvisorSuggestion | None = None\\n    capture: dict = Field(default_factory=dict)\\n    primary_limits: dict = Field(default_factory=dict)\\n    secondary_limits: dict = Field(default_factory=dict)\\n\\n\\ndef build_prompt(snapshot):\\n    return (\\n        snapshot.model_dump_json()\\n        + \\"\\\\nrl_advisor=\\"\\n        + (snapshot.rl_advisor.model_dump_json() if snapshot.rl_advisor else \\"none\\")\\n    )\\n\\n\\nclass ReasoningAgent(ABC):\\n    last_tools: list[dict] = []\\n\\n    @abstractmethod\\n    async def reason(self, input_data) -> tuple[LlmSubgoal, str]:\\n        pass\\n\\n\\nclass HeuristicAgent(ReasoningAgent):\\n    \\"\\"\\"Deterministic planner: proxy for a real LLM, and the fallback path.\\n\\n    Strategy:\\n      * far rendezvous: coast toward a capture point ~60% of current range at a\\n        capped closing speed.\\n      * closing/final approach: fly straight toward the port with a closing speed\\n        proportional to sqrt(range), capped per phase.\\n      * proximity: approach the port axially with a very small closing speed.\\n      * abort (priority change): hold a safe offset and brake to zero velocity.\\n    \\"\\"\\"\\n\\n    def __init__(self) -> None:\\n        self.phase_speed_caps = {\\n            DockingPhase.FAR_RENDEZVOUS.value: 1.5,  # m/s\\n            DockingPhase.CLOSING.value: 2.0,\\n            DockingPhase.FINAL_APPROACH.value: 0.5,\\n            # Below the dock velocity tolerance (0.1 m/s) so the final approach\\n            # never blows through the port faster than the docked condition.\\n            DockingPhase.PROXIMITY.value: 0.05,\\n            DockingPhase.DOCKED.value: 0.0,\\n        }\\n\\n    def plan(self, input_data: ReasoningInput) -> LlmSubgoal:\\n        \\"\\"\\"Synchronous deterministic planner (also used by the RL env).\\n\\n        In hybrid mode (``input_data.rl_advisor`` set) it *adopts* the RL\\n        policy\'s advisory suggestion — capped to the phase closing-speed and\\n        only when it is safe (reduces range, finite). Abort always overrides\\n        the advisor: the policy knows nothing about a priority change.\\n        \\"\\"\\"\\n        phase = input_data.phase\\n        r = np.asarray(input_data.position_m, dtype=float)\\n        current_vel = np.asarray(input_data.velocity_mps, dtype=float)\\n\\n        if np.linalg.norm(r) < 1e-9:\\n            target_pos = np.zeros(3)\\n            target_vel = np.zeros(3)\\n            closing = 0.0\\n            rationale = \\"Already at the docking port.\\"\\n        elif input_data.priority_change == \\"abort\\":\\n            # Safe hold: keep the *current* position fixed (a moving half-range\\n            # target would converge to the port) and brake nearly all relative\\n            # velocity. Never adopt the RL advisor on abort.\\n            target_pos = r.copy()\\n            target_vel = -current_vel * 0.9  # brake nearly all relative velocity\\n            closing = 0.0\\n            rationale = (\\n                \\"ABORT: mission priority changed; braking to a safe hold at current position.\\"\\n            )\\n        else:\\n            if any(\\n                \\"restored\\" not in a.detail and a.type.value == \\"sensor_dropout\\"\\n                for a in input_data.anomaly_context\\n            ):\\n                return LlmSubgoal(\\n                    target_position_m=r.tolist(),\\n                    target_velocity_mps=[0.0, 0.0, 0.0],\\n                    closing_speed_mps=0.0,\\n                    rationale=\\"Sensor reading stale: hold and brake until readings are restored.\\",\\n                )\\n            adopted = self._adopt_rl_advisor(input_data, phase)\\n            if adopted is not None:\\n                return adopted\\n            target_pos = r * (0.6 if phase == DockingPhase.FAR_RENDEZVOUS.value else 0.5)\\n            closing = min(\\n                self.phase_speed_caps.get(phase, 1.0),\\n                float(0.25 * np.sqrt(np.linalg.norm(r))),\\n            )\\n            target_vel = -target_pos / max(np.linalg.norm(target_pos), 1e-9) * closing\\n            rationale = (\\n                f\\"Phase {phase}: close toward port at {closing:.3f} m/s \\"\\n                f\\"(range {np.linalg.norm(r):.1f} m).\\"\\n            )\\n\\n        return LlmSubgoal(\\n            target_position_m=[float(v) for v in target_pos],\\n            target_velocity_mps=[float(v) for v in target_vel],\\n            closing_speed_mps=float(closing),\\n            rationale=rationale,\\n        )\\n\\n    def _adopt_rl_advisor(self, input_data: ReasoningInput, phase: str) -> LlmSubgoal | None:\\n        \\"\\"\\"Adopt the RL advisor\'s suggestion if it is safe; else None (fall back).\\"\\"\\"\\n        sugg = input_data.rl_advisor\\n        if sugg is None:\\n            return None\\n        r = np.asarray(input_data.position_m, dtype=float)\\n        target_pos = np.asarray(sugg.suggested_position_m, dtype=float)\\n        target_vel = np.asarray(sugg.suggested_velocity_mps, dtype=float)\\n        current_range = float(np.linalg.norm(r))\\n        suggested_range = float(np.linalg.norm(target_pos))\\n\\n        # Safety gate: the suggestion must reduce (or hold) range and be finite.\\n        if not (np.all(np.isfinite(target_pos)) and np.all(np.isfinite(target_vel))):\\n            return None\\n        if suggested_range >= current_range or np.dot(target_vel, r) >= 0:\\n            return None\\n        # Use learned velocity advice with a waypoint that still leads to the port.\\n        target_pos = r * 0.5\\n\\n        cap = self.phase_speed_caps.get(phase, 1.0)\\n        v_norm = float(np.linalg.norm(target_vel))\\n        if v_norm > cap:\\n            target_vel = target_vel / v_norm * cap\\n        closing = min(float(np.linalg.norm(target_vel)), sugg.closing_speed_mps)\\n        rationale = (\\n            f\\"RL advisor adopted (lookahead {sugg.lookahead_s:.0f}s): follow policy \\"\\n            f\\"suggestion at {closing:.3f} m/s (range {current_range:.1f} -> {suggested_range:.1f} m).\\"\\n        )\\n        return LlmSubgoal(\\n            target_position_m=[float(v) for v in target_pos],\\n            target_velocity_mps=[float(v) for v in target_vel],\\n            closing_speed_mps=float(closing),\\n            rationale=rationale,\\n        )\\n\\n    async def reason(self, input_data: ReasoningInput) -> tuple[LlmSubgoal, str]:\\n        return self.plan(input_data), \\"heuristic\\"\\n\\n\\ndef provider_has_key(provider):\\n    if provider == \\"heuristic\\":\\n        return True\\n    from dotenv import dotenv_values\\n\\n    dotenv = dotenv_values(\\".env\\")\\n    return any(\\n        os.environ.get(key) or dotenv.get(key) for key in PROVIDER_ENV_KEYS.get(provider, ())\\n    )\\n\\n\\nclass LangChainAgent(ReasoningAgent):\\n    \\"\\"\\"A small tool loop: inspect, preview, and submit one structured maneuver.\\"\\"\\"\\n\\n    def __init__(self, provider, model=None):\\n        from dotenv import dotenv_values\\n        from langchain_openai import ChatOpenAI\\n\\n        settings = get_settings()\\n        dotenv = dotenv_values(\\".env\\")\\n        key = next(\\n            (\\n                os.environ.get(k) or dotenv.get(k)\\n                for k in PROVIDER_ENV_KEYS[provider]\\n                if os.environ.get(k) or dotenv.get(k)\\n            ),\\n            None,\\n        )\\n        if not key:\\n            raise ValueError(f\\"No server-side API key configured for {provider}\\")\\n        self.provider = provider\\n        self.last_tools = []\\n        defaults = {\\n            \\"openai\\": settings.llm_model,\\n            \\"groq\\": settings.llm_model_groq,\\n            \\"openrouter\\": settings.llm_model_openrouter,\\n            \\"anthropic\\": \\"claude-sonnet-4-5\\",\\n        }\\n        if provider == \\"anthropic\\":\\n            from langchain_anthropic import ChatAnthropic\\n\\n            self.chat = ChatAnthropic(\\n                model=model or defaults[provider],\\n                api_key=key,\\n                timeout=settings.llm_timeout_s,\\n                max_retries=0,\\n            )\\n        else:\\n            self.chat = ChatOpenAI(\\n                model=model or defaults[provider],\\n                api_key=key,\\n                base_url=settings.llm_base_url or PROVIDER_BASE_URLS[provider],\\n                timeout=settings.llm_timeout_s,\\n                max_retries=0,\\n            )\\n\\n    async def reason(self, input_data):\\n        from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage\\n        from langchain_core.tools import StructuredTool\\n\\n        from app.llm.tools import docking_tools\\n\\n        def submit_maneuver(\\n            target_position_m, target_velocity_mps, closing_speed_mps, rationale, alternatives=None\\n        ):\\n            return LlmSubgoal(\\n                target_position_m=target_position_m,\\n                target_velocity_mps=target_velocity_mps,\\n                closing_speed_mps=closing_speed_mps,\\n                rationale=rationale,\\n                alternatives=alternatives or [],\\n            )\\n\\n        submit = StructuredTool.from_function(\\n            submit_maneuver,\\n            name=\\"submit_maneuver\\",\\n            args_schema=LlmSubgoal,\\n            description=\\"Submit the next bounded maneuver and a concise explanation with rejected alternatives.\\",\\n        )\\n        tools = docking_tools(input_data)\\n        lookup = {t.name: t for t in tools}\\n        self.last_tools = [{\\"name\\": t.name, \\"result\\": t.invoke({})} for t in tools[:4]]\\n        chat = self.chat.bind_tools(tools + [submit])\\n        messages = [\\n            SystemMessage(\\n                content=(\\n                    \\"Plan a satellite docking subgoal in a local frame: X/Y horizontal, Z vertical. \\"\\n                    \\"Use the read-only tools to inspect separation, capture, motion limits and learned PPO advice. \\"\\n                    \\"Submit one maneuver. Reduce separation, brake before capture, and respect both envelopes. \\"\\n                    \\"A positive minimum speed is only a transit preference; allow zero for braking and docking. \\"\\n                    \\"For abort hold current position and brake. For sensor loss brake conservatively. \\"\\n                    \\"Compare learned advice against limits and explain adoption or rejection. \\"\\n                    \\"Give a short decision justification and rejected alternatives, without private deliberation. \\"\\n                    \\"Capture is independently gated at at most 1.25 m. Do not claim capture yourself.\\"\\n                )\\n            ),\\n            HumanMessage(\\n                content=build_prompt(input_data) + \\"\\\\nTool readings=\\" + json.dumps(self.last_tools)\\n            ),\\n        ]\\n        for _ in range(4):\\n            response = await chat.ainvoke(messages)\\n            messages.append(response)\\n            if not response.tool_calls:\\n                raise ValueError(\\"Provider did not return a tool call\\")\\n            result = None\\n            for call in response.tool_calls:\\n                if call[\\"name\\"] == submit.name:\\n                    result = LlmSubgoal.model_validate(call[\\"args\\"])\\n                    output = {\\"accepted\\": True}\\n                elif call[\\"name\\"] in lookup:\\n                    output = lookup[call[\\"name\\"]].invoke(call[\\"args\\"])\\n                    self.last_tools.append(\\n                        {\\"name\\": call[\\"name\\"], \\"arguments\\": call[\\"args\\"], \\"result\\": output}\\n                    )\\n                else:\\n                    output = {\\"error\\": \\"Unknown tool\\"}\\n                messages.append(ToolMessage(content=json.dumps(output), tool_call_id=call[\\"id\\"]))\\n            if result is not None:\\n                return result, self.provider\\n        raise ValueError(\\"Provider exhausted the four-round tool budget\\")\\n\\n\\nclass ReasoningAgentFactory:\\n    @staticmethod\\n    def build(provider=None, fallback_provider=None, model=None):\\n        provider = provider or get_settings().mode\\n        if provider == \\"heuristic\\":\\n            return HeuristicAgent()\\n        if provider not in PROVIDER_ENV_KEYS:\\n            raise ValueError(f\\"Unknown provider: {provider}\\")\\n        return LangChainAgent(provider, model)\\n\\n    @staticmethod\\n    def heuristic():\\n        return HeuristicAgent()\\n\\n\\nasync def with_retry_heuristic_fallback(\\n    agent, input_data, heuristic, max_retries=2, backoff_s=0.25\\n):\\n    for attempt in range(max_retries + 1):\\n        try:\\n            return await asyncio.wait_for(agent.reason(input_data), get_settings().llm_timeout_s)\\n        except Exception:\\n            if attempt < max_retries:\\n                await asyncio.sleep(backoff_s * (attempt + 1))\\n    subgoal = heuristic.plan(input_data)\\n    subgoal.rationale = (\\n        \\"Provider unavailable or invalid response; deterministic fallback. \\" + subgoal.rationale\\n    )\\n    return subgoal, \\"heuristic_fallback\\"\\n", "src/app/llm/rl_advisor.py": "\\"\\"\\"RL advisor: the trained policy as informed context for the LLM planner.\\n\\nThe LLM-reasoning agent (or the deterministic heuristic fallback) is the\\ndecision-maker, but when ``config.rl_advisor_enabled`` is set it receives an\\n*advisory sub-goal* computed from the trained RL policy at every replan:\\n\\n  1. Run the policy on the current (state, active sub-goal) — it commands a\\n     bounded acceleration, exactly as it does when used as ``controller=\\"rl\\"``.\\n  2. Project that acceleration forward by ``lookahead_s`` into a suggested\\n     target position/velocity — \\"where the learned policy is heading\\".\\n  3. Hand the suggestion to the reasoning layer as context; the planner may\\n     adopt it (optionally capped), override it, or reject it, and its\\n     rationale records which it did.\\n\\nThis is the hybrid research mode: the LLM keeps maneuver interpretability\\n(every decision carries a natural-language justification) while its planning\\nis informed by what the RL policy has learned. The suggestion is also echoed\\non each ``llm_decision`` event, so the stream shows \\"RL suggested X, planner\\nchose Y\\".\\n\\nThe RL artifacts (model + VecNormalize stats) are the same ones used by\\n``controller=\\"rl\\"``; if they\'re absent or the policy produces a non-finite\\ncommand, ``suggest`` returns ``None`` and the planner simply proceeds without\\nadvisory context.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport logging\\n\\nimport numpy as np\\n\\nfrom app.config import get_settings\\nfrom app.schemas.events import LlmSubgoal, RlAdvisorSuggestion\\nfrom app.simulation.dynamics import RigidBodyState\\n\\nlogger = logging.getLogger(\\"llm_docking.rl_advisor\\")\\n\\n\\nclass RlAdvisor:\\n    \\"\\"\\"Runs the trained policy at replan time and produces a suggestion.\\"\\"\\"\\n\\n    def __init__(\\n        self,\\n        mass_kg: float = 500.0,\\n        inertia: np.ndarray | None = None,\\n        lookahead_s: float = 4.0,\\n        model_path: str | None = None,\\n        normalize_path: str | None = None,\\n    ) -> None:\\n        from app.simulation.dynamics import DEFAULT_INERTIA\\n\\n        self.mass_kg = float(mass_kg)\\n        self.inertia = np.asarray(inertia if inertia is not None else DEFAULT_INERTIA, dtype=float)\\n        self.lookahead_s = float(lookahead_s)\\n        settings = get_settings()\\n        self.model_path = model_path or settings.rl_model_path\\n        self.normalize_path = normalize_path or settings.rl_normalize_path\\n        self._controller = None  # lazy: RlController pulls in the RL extras\\n        self._load_error: str | None = None\\n\\n    def suggest(\\n        self,\\n        state: RigidBodyState,\\n        subgoal: LlmSubgoal | None,\\n    ) -> RlAdvisorSuggestion | None:\\n        \\"\\"\\"Return the policy\'s advisory target, or None when unavailable.\\n\\n        Never raises: a missing/broken policy degrades to no-advice, not a\\n        failed run.\\n        \\"\\"\\"\\n        controller = self._get_controller()\\n        if controller is None or subgoal is None:\\n            return None\\n\\n        from app.simulation.controllers.base import ControlContext\\n\\n        try:\\n            cmd = controller.compute(state, subgoal, dt=1.0, ctx=ControlContext())\\n        except Exception:  # noqa: BLE001 - advisory only; never crash the sim\\n            logger.warning(\\n                \\"rl_advisor_failed\\",\\n                extra={\\"error\\": str(self._load_error or \\"policy compute failed\\")},\\n            )\\n            return None\\n\\n        accel = np.asarray(cmd.force_n, dtype=float) / self.mass_kg\\n        if not np.all(np.isfinite(accel)):\\n            logger.warning(\\"rl_advisor_non_finite_command\\")\\n            return None\\n\\n        tau = self.lookahead_s\\n        v_sugg = np.asarray(state.velocity, dtype=float) + accel * tau\\n        p_sugg = (\\n            np.asarray(state.position, dtype=float) + state.velocity * tau + 0.5 * accel * tau**2\\n        )\\n\\n        p_norm = float(np.linalg.norm(p_sugg))\\n        closing = 0.0\\n        if p_norm > 1e-9:\\n            closing = max(0.0, -float(np.dot(v_sugg, p_sugg) / p_norm))\\n\\n        return RlAdvisorSuggestion(\\n            suggested_position_m=[float(v) for v in p_sugg],\\n            suggested_velocity_mps=[float(v) for v in v_sugg],\\n            closing_speed_mps=closing,\\n            lookahead_s=tau,\\n        )\\n\\n    # ------------------------------------------------------------------\\n\\n    def _get_controller(self):\\n        if self._controller is not None:\\n            return self._controller\\n        if self._load_error is not None:\\n            return None\\n        if not self.model_path or not self.normalize_path:\\n            self._load_error = (\\n                \\"RL artifacts not configured (LLM_DOCKING_RL_MODEL_PATH/NORMALIZE_PATH)\\"\\n            )\\n            logger.warning(\\"rl_advisor_disabled\\", extra={\\"reason\\": self._load_error})\\n            return None\\n        try:\\n            from app.simulation.controllers.rl import RlController\\n\\n            self._controller = RlController(\\n                mass_kg=self.mass_kg,\\n                inertia=self.inertia,\\n                model_path=self.model_path,\\n                normalize_path=self.normalize_path,\\n            )\\n            return self._controller\\n        except Exception as exc:  # noqa: BLE001 - degrade to no-advice\\n            self._load_error = str(exc)\\n            logger.warning(\\"rl_advisor_load_failed\\", extra={\\"error\\": self._load_error})\\n            return None\\n", "src/app/llm/tools.py": "\\"\\"\\"Read-only geometry tools. Only the runner can apply actuator commands.\\"\\"\\"\\n\\nimport numpy as np\\nfrom langchain_core.tools import tool\\n\\n\\ndef docking_tools(snapshot):\\n    \\"\\"\\"Bind tools to one fresh state snapshot; the model cannot alter telemetry.\\"\\"\\"\\n\\n    @tool\\n    def measure_distance() -> dict:\\n        \\"\\"\\"Measure docking-port separation, relative speed, and closing speed in SI units.\\"\\"\\"\\n        position = np.asarray(snapshot.position_m)\\n        velocity = np.asarray(snapshot.velocity_mps)\\n        distance = float(np.linalg.norm(position))\\n        return {\\n            \\"distance_m\\": distance,\\n            \\"relative_speed_mps\\": float(np.linalg.norm(velocity)),\\n            \\"closing_speed_mps\\": -float(np.dot(position, velocity)) / max(distance, 1e-9),\\n        }\\n\\n    @tool\\n    def inspect_capture_gate() -> dict:\\n        \\"\\"\\"Read the shared 1.25 metre gate and speed, attitude, rotation checks.\\"\\"\\"\\n        return snapshot.capture\\n\\n    @tool\\n    def inspect_actuators() -> dict:\\n        \\"\\"\\"Read thruster limits and the configured primary and secondary motion envelopes.\\"\\"\\"\\n        return {\\n            \\"force_per_axis_n\\": 50.0,\\n            \\"torque_per_axis_nm\\": 20.0,\\n            \\"primary_limits\\": snapshot.primary_limits,\\n            \\"secondary_limits\\": snapshot.secondary_limits,\\n            \\"minimum_speed_rule\\": \\"Transit preference; braking and capture may use zero speed.\\",\\n        }\\n\\n    @tool\\n    def inspect_rl_heuristic() -> dict:\\n        \\"\\"\\"Read the learned PPO policy proposal, if a trained model is configured.\\"\\"\\"\\n        return {\\n            \\"available\\": snapshot.rl_advisor is not None,\\n            \\"proposal\\": snapshot.rl_advisor.model_dump() if snapshot.rl_advisor else None,\\n        }\\n\\n    @tool\\n    def preview_maneuver(target_position_m: list[float], target_velocity_mps: list[float]) -> dict:\\n        \\"\\"\\"Compare a proposed target with current range and configured speed limits without moving.\\"\\"\\"\\n        if len(target_position_m) != 3 or len(target_velocity_mps) != 3:\\n            return {\\"error\\": \\"Positions and velocities must have three components\\"}\\n        if not np.isfinite(target_position_m + target_velocity_mps).all():\\n            return {\\"error\\": \\"Components must be finite\\"}\\n        return {\\n            \\"target_range_m\\": float(np.linalg.norm(target_position_m)),\\n            \\"reduces_range\\": bool(np.linalg.norm(target_position_m) < snapshot.range_m),\\n            \\"target_speed_mps\\": float(np.linalg.norm(target_velocity_mps)),\\n        }\\n\\n    return [\\n        measure_distance,\\n        inspect_capture_gate,\\n        inspect_actuators,\\n        inspect_rl_heuristic,\\n        preview_maneuver,\\n    ]\\n", "src/app/llm/__init__.py": "\\"\\"\\"LLM reasoning layer: a high-level planner that emits structured sub-goals.\\"\\"\\"\\n", "src/app/schemas/docking.py": "\\"\\"\\"Request, state and summary schemas for the docking API.\\n\\nUnits are explicit and consistent everywhere:\\n  * positions  : meters (m), Hill/RSW frame, origin at the primary satellite\\n  * velocities : meters/second (m/s)\\n  * quaternions: [w, x, y, z] (w = scalar part first)\\n  * angular    : radians/second (rad/s)\\n  * fuel       : percentage remaining [0..100]\\n  * time       : seconds (s) for the step interval; total elapsed sim time in s\\nFields are validated so a malformed request fails fast with a 422 before a\\nsession is created.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport math\\nfrom enum import StrEnum\\nfrom typing import Annotated, Literal\\n\\nfrom pydantic import BaseModel, ConfigDict, Field, model_validator\\n\\n# ---------------------------------------------------------------------------\\n# Primitive types\\n# ---------------------------------------------------------------------------\\n\\n\\ndef _valid_unit_quaternion(v: list[float]) -> bool:\\n    if len(v) != 4:\\n        return False\\n    if not all(isinstance(x, (int, float)) and math.isfinite(x) for x in v):\\n        return False\\n    return abs(math.sqrt(sum(x * x for x in v)) - 1.0) < 1e-3\\n\\n\\nVector3 = Annotated[\\n    list[float],\\n    Field(min_length=3, max_length=3, description=\\"3-vector, e.g. position [x,y,z] in meters\\"),\\n]\\n\\nQuaternion = Annotated[\\n    list[float],\\n    Field(min_length=4, max_length=4, description=\\"Unit quaternion [w,x,y,z]\\"),\\n]\\n\\n\\nclass ControllerType(StrEnum):\\n    \\"\\"\\"Selectable low-level controllers.\\"\\"\\"\\n\\n    PID = \\"pid\\"\\n    LQR = \\"lqr\\"\\n    RL = \\"rl\\"  # trained Stable-Baselines3 PPO policy (see app/simulation/rl/)\\n\\n\\nclass AnomalyType(StrEnum):\\n    \\"\\"\\"Supported injectable anomalies.\\n\\n    * SENSOR_DROPOUT  -> corrupt or withhold a sensor reading for `duration_steps`\\n    * VELOCITY_IMPULSE -> apply an unmodeled velocity impulse (m/s)\\n    * PRIORITY_CHANGE  -> inject a \\"priority change / abort\\" instruction that the\\n                          LLM layer must interpret (it is *not* fed to the controller)\\n    \\"\\"\\"\\n\\n    SENSOR_DROPOUT = \\"sensor_dropout\\"\\n    VELOCITY_IMPULSE = \\"velocity_impulse\\"\\n    PRIORITY_CHANGE = \\"priority_change\\"\\n\\n\\nclass DockingPhase(StrEnum):\\n    \\"\\"\\"Docking phase classification by relative range (see README).\\"\\"\\"\\n\\n    FAR_RENDEZVOUS = \\"far_rendezvous\\"  # > 1000 m\\n    CLOSING = \\"closing\\"  # 1000 – 30 m\\n    FINAL_APPROACH = \\"final_approach\\"  # 30 – 5 m\\n    PROXIMITY = \\"proximity\\"  # 5 – 0 m\\n    DOCKED = \\"docked\\"  # within position + attitude tolerance\\n\\n\\nclass FailureReason(StrEnum):\\n    CANCELLED = \\"cancelled\\"\\n    LIMIT_EXCEEDED = \\"limit_exceeded\\"\\n    FUEL_EXHAUSTED = \\"fuel_exhausted\\"\\n    MAX_STEPS_EXCEEDED = \\"max_steps_exceeded\\"\\n    MAX_TIME_EXCEEDED = \\"max_time_exceeded\\"\\n    DIVERGED = \\"diverged\\"  # relative range exceeded max_range_m (unstable/unrecoverable)\\n    UNRECOVERABLE_ANOMALY = \\"unrecoverable_anomaly\\"\\n\\n\\nclass StopReason(StrEnum):\\n    DOCKED = \\"docked\\"\\n    FAILED = \\"failed\\"\\n\\n\\n# ---------------------------------------------------------------------------\\n# Orbital / state schemas\\n# ---------------------------------------------------------------------------\\n\\n\\nclass OrbitalElements(BaseModel):\\n    \\"\\"\\"Classical orbital elements of a satellite; used to derive mean motion n.\\n\\n    ``mean_motion_radps`` is optional — if present it overrides the value\\n    computed from ``semi_major_axis_m`` + gravitational parameter. This lets\\n    callers model unusual orbits (e.g. non-circular) without changing physics.\\n    \\"\\"\\"\\n\\n    model_config = ConfigDict(extra=\\"forbid\\", allow_inf_nan=False)\\n    semi_major_axis_m: float = Field(gt=0, description=\\"Semi-major axis of the orbit [m]\\")\\n    eccentricity: float = Field(default=0.0, ge=0.0, lt=1.0)\\n    inclination_deg: float = Field(default=0.0, ge=-90.0, le=180.0)\\n    raan_deg: float = Field(default=0.0)\\n    arg_periapsis_deg: float = Field(default=0.0)\\n    true_anomaly_deg: float = Field(default=0.0)\\n    mean_motion_radps: float | None = Field(\\n        default=None, gt=0.0, description=\\"Optional explicit mean motion [rad/s]\\"\\n    )\\n\\n\\nclass MotionLimits(BaseModel):\\n    \\"\\"\\"Speed limits in the local frame; Z is vertical, XY is horizontal.\\"\\"\\"\\n\\n    model_config = ConfigDict(extra=\\"forbid\\", allow_inf_nan=False)\\n    min_speed_mps: float = Field(default=0.0, ge=0.0)\\n    max_speed_mps: float = Field(default=2.0, gt=0.0, le=20.0)\\n    max_horizontal_speed_mps: float = Field(default=2.0, gt=0.0, le=20.0)\\n    max_vertical_speed_mps: float = Field(default=1.0, gt=0.0, le=20.0)\\n\\n    @model_validator(mode=\\"after\\")\\n    def validate_minimum(self):\\n        if self.min_speed_mps > min(self.max_speed_mps, self.max_horizontal_speed_mps):\\n            raise ValueError(\\"Minimum transit speed must fit the speed envelope\\")\\n        return self\\n\\n\\nclass SatelliteState(BaseModel):\\n    \\"\\"\\"Full inertial-ish state of one satellite.\\n\\n    Positions/velocities are the *relative Hill/RSW* state when used for the\\n    secondary (chaser) — i.e. relative to the primary — while the primary is\\n    always at [0,0,0] in the Hill frame. The primary additionally carries an\\n    inertial position (ECI) that the future frontend can use to anchor the\\n    camera/orbit rendering.\\n    \\"\\"\\"\\n\\n    model_config = ConfigDict(extra=\\"forbid\\", allow_inf_nan=False)\\n\\n    position_m: Vector3 = Field(default_factory=lambda: [0.0, 0.0, 0.0])\\n    velocity_mps: Vector3 = Field(default_factory=lambda: [0.0, 0.0, 0.0])\\n    orientation_quat: Quaternion = Field(default_factory=lambda: [1.0, 0.0, 0.0, 0.0])\\n    angular_velocity_radps: Vector3 = Field(default_factory=lambda: [0.0, 0.0, 0.0])\\n    fuel_remaining_pct: float = Field(default=100.0, ge=0.0, le=100.0)\\n    orbital_elements: OrbitalElements | None = None\\n    inertia_kg_m2: list[float] | None = Field(\\n        default=None,\\n        min_length=3,\\n        max_length=3,\\n        description=\\"Principal moments of inertia [Ix, Iy, Iz] in kg·m² (defaults used if unset)\\",\\n    )\\n    mass_kg: float = Field(default=500.0, gt=0.0)\\n    name: str = Field(default=\\"Spacecraft\\", min_length=1, max_length=80)\\n    capture_radius_m: float = Field(default=1.25, gt=0.0, le=1.25)\\n    motion_limits: MotionLimits = Field(default_factory=MotionLimits)\\n    # Inertial (ECI) position of the primary, for orbit/camera rendering.\\n    inertial_position_m: Vector3 | None = Field(default=None, min_length=3, max_length=3)\\n\\n    @model_validator(mode=\\"after\\")\\n    def _validate(self) -> SatelliteState:\\n        if not _valid_unit_quaternion(self.orientation_quat):\\n            raise ValueError(\\"orientation_quat must be a unit quaternion (norm within 1e-3 of 1)\\")\\n        if self.inertia_kg_m2 is not None and any(v <= 0 for v in self.inertia_kg_m2):\\n            raise ValueError(\\"Moments of inertia must be positive\\")\\n        return self\\n\\n\\n# ---------------------------------------------------------------------------\\n# Simulation config\\n# ---------------------------------------------------------------------------\\n\\n\\nclass AnomalyConfig(BaseModel):\\n    \\"\\"\\"A single anomaly to inject. ``None`` per type means \\"not injected\\".\\"\\"\\"\\n\\n    model_config = ConfigDict(extra=\\"forbid\\", allow_inf_nan=False)\\n\\n    type: AnomalyType\\n    step: int = Field(ge=0, description=\\"Physics step index at which the anomaly fires\\")\\n    detail: str = Field(\\n        default=\\"\\",\\n        max_length=500,\\n        description=\\"Free-form detail surfaced in the anomaly event + LLM context\\",\\n    )\\n\\n    # sensor_dropout\\n    duration_steps: int | None = Field(default=None, ge=1)\\n    corrupt: bool = Field(\\n        default=True,\\n        description=\\"If False, withhold the reading (gap); if True, corrupt it with noise\\",\\n    )\\n\\n    # velocity_impulse\\n    impulse_mps: list[float] | None = Field(default=None, min_length=3, max_length=3)\\n\\n    # priority_change\\n    priority_change: Literal[\\"abort\\"] | None = Field(\\n        default=None, description=\\"\'abort\' means the LLM is told the mission is aborted\\"\\n    )\\n\\n    @model_validator(mode=\\"after\\")\\n    def _validate(self) -> AnomalyConfig:\\n        if self.type is AnomalyType.VELOCITY_IMPULSE and self.impulse_mps is None:\\n            raise ValueError(\\"velocity_impulse requires impulse_mps [vx,vy,vz]\\")\\n        if self.type is AnomalyType.SENSOR_DROPOUT and self.duration_steps is None:\\n            raise ValueError(\\"sensor_dropout requires duration_steps\\")\\n        return self\\n\\n\\nclass SimulationConfig(BaseModel):\\n    model_config = ConfigDict(extra=\\"forbid\\", allow_inf_nan=False)\\n\\n    controller: ControllerType = ControllerType.LQR\\n    step_interval_s: float = Field(default=1.0, gt=0.0, le=1.0)\\n    max_steps: int = Field(default=600, ge=1, le=100_000)\\n    max_time_s: float | None = Field(default=None, gt=0.0)\\n    llm_enabled: bool = True\\n    llm_cadence_steps: int = Field(\\n        default=10, ge=1, le=1000, description=\\"Re-plan every N physics steps\\"\\n    )\\n    # LLM x RL hybrid: when true, each replan feeds the reasoning layer an\\n    # advisory sub-goal computed from the trained RL policy (requires the same\\n    # LLM_DOCKING_RL_MODEL_PATH / _NORMALIZE_PATH artifacts as controller=\\"rl\\").\\n    rl_advisor_enabled: bool = Field(\\n        default=False,\\n        description=\\"The LLM planner receives a suggestion from the trained RL policy\\",\\n    )\\n    rl_advisor_lookahead_s: float = Field(\\n        default=4.0,\\n        gt=0.0,\\n        le=60.0,\\n        description=\\"How far ahead the RL advisor projects its commanded acceleration\\",\\n    )\\n    anomalies: list[AnomalyConfig] = Field(default_factory=list, max_length=10)\\n    # Tolerances for the \\"docked\\" condition.\\n    dock_position_tol_m: float = Field(default=1.25, gt=0.0, le=1.25)\\n    dock_velocity_tol_mps: float = Field(default=0.1, gt=0.0)\\n    dock_attitude_tol_deg: float = Field(default=2.0, gt=0.0, le=90.0)\\n    # Fuel: normalized mass-flow per second of thruster use.\\n    fuel_consumption_rate_pct_per_s: float = Field(default=0.05, ge=0.0)\\n    # Divergence guards: fail the run (FailureReason.DIVERGED) if the relative\\n    # range exceeds ``max_range_m`` or the body rate exceeds ``max_omega_radps``\\n    # (an unstable policy — e.g. a bad RL checkpoint — can fling the chaser or\\n    # tumble it; both also protect the integrator from numerical blow-up).\\n    max_range_m: float | None = Field(default=10_000.0, gt=0.0)\\n    max_omega_radps: float = Field(default=5.0, gt=0.0)\\n    # Optional wall-clock delay between SSE events (demo pacing for a watching\\n    # frontend; 0 = stream as fast as the sim produces events).\\n    stream_pacing_s: float = Field(default=0.0, ge=0.0, le=1.0)\\n    seed: int | None = Field(default=None, description=\\"RNG seed for reproducible noise/impulses\\")\\n    provider: Literal[\\"heuristic\\", \\"openai\\", \\"openrouter\\", \\"groq\\", \\"anthropic\\"] | None = None\\n    model: str | None = Field(default=None, min_length=1, max_length=120)\\n    auto_dock: bool = True\\n    initial_separation_range_m: tuple[float, float] | None = Field(\\n        default=None,\\n        description=\\"Optional inclusive starting port-distance bounds [minimum, maximum] in metres\\",\\n    )\\n\\n\\n# ---------------------------------------------------------------------------\\n# Request / response\\n# ---------------------------------------------------------------------------\\n\\n\\nclass SimulateRequest(BaseModel):\\n    \\"\\"\\"POST /api/v1/docking/simulate body.\\"\\"\\"\\n\\n    model_config = ConfigDict(extra=\\"forbid\\")\\n\\n    primary: SatelliteState\\n    secondary: SatelliteState\\n    config: SimulationConfig\\n\\n    @model_validator(mode=\\"after\\")\\n    def _validate(self) -> SimulateRequest:\\n        if self.config.initial_separation_range_m is not None:\\n            minimum, maximum = self.config.initial_separation_range_m\\n            if not 0 < minimum <= maximum:\\n                raise ValueError(\\"Initial separation bounds must satisfy 0 < minimum <= maximum\\")\\n            separation = math.dist(self.primary.position_m, self.secondary.position_m)\\n            if not minimum <= separation <= maximum:\\n                raise ValueError(\\n                    f\\"Initial port separation must be between {minimum:g} and {maximum:g} m \\"\\n                    f\\"(received {separation:.3f} m)\\"\\n                )\\n        for satellite in [self.primary, self.secondary]:\\n            speed = math.sqrt(sum(v * v for v in satellite.velocity_mps))\\n            horizontal = math.hypot(*satellite.velocity_mps[:2])\\n            limits = satellite.motion_limits\\n            if (\\n                speed > limits.max_speed_mps\\n                or horizontal > limits.max_horizontal_speed_mps\\n                or abs(satellite.velocity_mps[2]) > limits.max_vertical_speed_mps\\n            ):\\n                raise ValueError(\\"Initial velocity exceeds spacecraft motion limits\\")\\n        if self.secondary.orbital_elements is None and self.primary.orbital_elements is not None:\\n            # Secondary inherits the primary\'s orbit (same orbital regime in Hill frame).\\n            self.secondary.orbital_elements = self.primary.orbital_elements\\n        if self.primary.orbital_elements is None and self.secondary.orbital_elements is None:\\n            raise ValueError(\\n                \\"At least one satellite must specify orbital_elements so mean motion n can be derived\\"\\n            )\\n        return self\\n\\n\\nclass SimulateResponse(BaseModel):\\n    \\"\\"\\"HTTP 202 response returned immediately; the SSE stream carries the rest.\\"\\"\\"\\n\\n    session_id: str\\n    stream_url: str\\n    summary_url: str\\n\\n\\n# ---------------------------------------------------------------------------\\n# Session summary (polling endpoint)\\n# ---------------------------------------------------------------------------\\n\\n\\nclass Metrics(BaseModel):\\n    \\"\\"\\"Accumulated cost/error metrics for the run (evaluation harness data).\\"\\"\\"\\n\\n    delta_v_mps: float = 0.0\\n    ise: float = 0.0  # integral of squared position error (m²·s)\\n    max_commanded_thrust_n: float = 0.0\\n    total_llm_calls: int = 0\\n    llm_fallback_calls: int = 0\\n    anomaly_events_count: int = 0\\n\\n\\nclass SessionSummary(BaseModel):\\n    \\"\\"\\"GET /api/v1/docking/sessions/{session_id} response.\\"\\"\\"\\n\\n    session_id: str\\n    status: Literal[\\"running\\", \\"docked\\", \\"failed\\"]\\n    stop_reason: StopReason | None = None\\n    failure_reason: FailureReason | None = None\\n    steps_completed: int = 0\\n    sim_time_s: float = 0.0\\n    phase: DockingPhase | None = None\\n    final_secondary: SatelliteState | None = None\\n    controller: ControllerType\\n    metrics: Metrics = Field(default_factory=Metrics)\\n    events_emitted: int = 0\\n    llm_enabled: bool = True\\n    error: str | None = None\\n", "src/app/schemas/events.py": "\\"\\"\\"Event schemas emitted on the SSE stream.\\n\\nEvery simulation produces an ordered stream of ``DockingEvent`` members — the\\nsingle wire contract the frontend and the eval harness consume. Events are\\ndiscriminated by their ``type`` field:\\n\\n  * ``telemetry``    — per-step observed state + the commanded thrust\\n  * ``heartbeat``    — periodic liveness marker while the run is active\\n  * ``llm_decision`` — a new LLM sub-goal (replan), with its source\\n  * ``anomaly``      — an injected anomaly fired or restored\\n  * ``docked``       — terminal success event (with final state + metrics)\\n  * ``failed``       — terminal failure event (with reason + metrics)\\n\\nEvery event carries a shared envelope (``StreamEventBase``): ``schema_version``\\n(so the frontend can assert compatibility before parsing), ``session_id`` (so a\\nclient multiplexing several runs can route events without external state), and\\nthe sim ``step`` / ``sim_time_s`` (so a reconnecting client can resume and\\nre-order). ``AnomalyDetail`` and ``LlmSubgoal`` are shared payloads also\\nembedded in the LLM prompt context and the controller interface respectively.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import Annotated, Literal\\n\\nfrom pydantic import BaseModel, ConfigDict, Field\\n\\nfrom app.schemas.docking import (\\n    AnomalyType,\\n    ControllerType,\\n    DockingPhase,\\n    FailureReason,\\n    Metrics,\\n    Quaternion,\\n    SatelliteState,\\n    Vector3,\\n)\\n\\n# ---------------------------------------------------------------------------\\n# Shared envelope\\n# ---------------------------------------------------------------------------\\n\\n\\nclass StreamEventBase(BaseModel):\\n    \\"\\"\\"Fields present on every stream event (schema version + routing/clock).\\"\\"\\"\\n\\n    schema_version: Literal[\\"1.0\\"] = \\"1.0\\"\\n    session_id: str = \\"\\"\\n    event_id: int = 0\\n    step: int = Field(ge=0)\\n    sim_time_s: float = Field(ge=0.0)\\n\\n\\nclass PrimaryFrame(BaseModel):\\n    \\"\\"\\"The primary satellite\'s frame, for frontend camera/orbit rendering.\\n\\n    The primary is the Hill-frame origin, so its relative position is always\\n    [0, 0, 0]; its attitude (and any inertial position) anchors the scene.\\n    \\"\\"\\"\\n\\n    position_m: Vector3 = Field(default_factory=lambda: [0.0, 0.0, 0.0])\\n    orientation_quat: Quaternion = Field(default_factory=lambda: [1.0, 0.0, 0.0, 0.0])\\n\\n\\n# ---------------------------------------------------------------------------\\n# Shared payloads\\n# ---------------------------------------------------------------------------\\n\\n\\nclass AnomalyDetail(BaseModel):\\n    \\"\\"\\"One anomaly occurrence, surfaced both on the stream and to the LLM.\\"\\"\\"\\n\\n    type: AnomalyType\\n    detail: str = Field(default=\\"\\", max_length=500)\\n    step: int = Field(ge=0)\\n    sensor: str | None = None\\n    impulse_mps: Vector3 | None = None\\n    priority_change: Literal[\\"abort\\"] | None = None\\n\\n\\nclass LlmSubgoal(BaseModel):\\n    \\"\\"\\"Structured sub-goal emitted by the reasoning layer (LLM or heuristic).\\n\\n    The low-level controller tracks ``target_position_m`` / ``target_velocity_mps``\\n    until the next replan; ``closing_speed_mps`` is the envelope magnitude the\\n    planner committed to, and ``rationale`` is the human-readable justification\\n    (this is the event\'s ``llm_reasoning_summary`` — the natural-language\\n    justification the reasoning layer produced).\\n    \\"\\"\\"\\n\\n    model_config = ConfigDict(extra=\\"forbid\\", allow_inf_nan=False)\\n    target_position_m: Vector3\\n    target_velocity_mps: Vector3\\n    closing_speed_mps: float = Field(ge=0.0)\\n    rationale: str = Field(default=\\"\\", max_length=500)\\n    alternatives: list[str] = Field(default_factory=list, max_length=4)\\n\\n\\nclass RlAdvisorSuggestion(BaseModel):\\n    \\"\\"\\"Advisory sub-goal proposed by the trained RL policy at a replan.\\n\\n    The policy is a *tracking* controller: given the current state and the\\n    active sub-goal it commands an acceleration, which we project forward by\\n    ``lookahead_s`` into a target position/velocity the policy is heading\\n    toward. The LLM planner receives this as informed context and may adopt\\n    it, cap it, or override it — its decision is what lands in\\n    ``llm_decision.subgoal``, and this payload is echoed on the event so the\\n    stream shows \\"RL suggested X, planner chose Y\\" (interpretability).\\n    \\"\\"\\"\\n\\n    model_config = ConfigDict(allow_inf_nan=False)\\n    suggested_position_m: Vector3\\n    suggested_velocity_mps: Vector3\\n    closing_speed_mps: float = Field(ge=0.0)\\n    lookahead_s: float = Field(gt=0.0)\\n    policy_source: Literal[\\"rl\\"] = \\"rl\\"\\n\\n\\n# ---------------------------------------------------------------------------\\n# Stream events\\n# ---------------------------------------------------------------------------\\n\\n\\nclass TelemetryEvent(StreamEventBase):\\n    \\"\\"\\"Per-step observed state and the thruster command applied.\\"\\"\\"\\n\\n    type: Literal[\\"telemetry\\"] = \\"telemetry\\"\\n    phase: DockingPhase\\n    position_m: Vector3\\n    velocity_mps: Vector3\\n    range_m: float = Field(ge=0.0)\\n    closing_speed_mps: float = Field(ge=0.0)\\n    fuel_pct: float = Field(ge=0.0, le=100.0)\\n    controller: ControllerType\\n    llm_source: str | None = None\\n    command_force_n: Vector3 | None = None\\n    command_torque_nm: Vector3 | None = None\\n    # Frontend anchors: primary frame (origin + attitude) and running metrics\\n    # so a live dashboard can plot Δv/ISE without waiting for the terminal event.\\n    primary: PrimaryFrame = Field(default_factory=PrimaryFrame)\\n    metrics_so_far: Metrics = Field(default_factory=Metrics)\\n    orientation_quat: Quaternion = Field(default_factory=lambda: [1.0, 0.0, 0.0, 0.0])\\n    angular_velocity_radps: Vector3 = Field(default_factory=lambda: [0.0, 0.0, 0.0])\\n    primary_velocity_mps: Vector3 = Field(default_factory=lambda: [0.0, 0.0, 0.0])\\n    secondary_velocity_mps: Vector3 = Field(default_factory=lambda: [0.0, 0.0, 0.0])\\n    capture: dict = Field(default_factory=dict)\\n    command_limited: bool = False\\n\\n\\nclass HeartbeatEvent(StreamEventBase):\\n    \\"\\"\\"Periodic liveness marker (wall-clock interval, not sim steps).\\"\\"\\"\\n\\n    type: Literal[\\"heartbeat\\"] = \\"heartbeat\\"\\n    status: Literal[\\"running\\", \\"docked\\", \\"failed\\"] = \\"running\\"\\n\\n\\nclass LlmDecisionEvent(StreamEventBase):\\n    \\"\\"\\"A replan: the reasoning layer (LLM or heuristic) emits a new sub-goal.\\"\\"\\"\\n\\n    type: Literal[\\"llm_decision\\"] = \\"llm_decision\\"\\n    phase: DockingPhase\\n    subgoal: LlmSubgoal\\n    source: str = Field(\\n        description=\\"openai | openrouter | groq | anthropic | heuristic | heuristic_fallback\\"\\n    )\\n    anomaly_context: list[AnomalyDetail] = Field(default_factory=list)\\n    priority_change: str | None = None\\n    metrics_so_far: Metrics = Field(default_factory=Metrics)\\n    # The RL policy\'s advisory suggestion the planner saw (when enabled), so\\n    # the stream shows what the RL agent proposed vs what the planner chose.\\n    rl_advisor: RlAdvisorSuggestion | None = None\\n    tool_results: list[dict] = Field(default_factory=list)\\n    safety_notes: list[str] = Field(default_factory=list)\\n\\n\\nclass AnomalyEvent(StreamEventBase):\\n    \\"\\"\\"An anomaly fired (or a sensor dropout window restored).\\"\\"\\"\\n\\n    type: Literal[\\"anomaly\\"] = \\"anomaly\\"\\n    anomaly: AnomalyDetail\\n\\n\\nclass DockedEvent(StreamEventBase):\\n    \\"\\"\\"Terminal success: within position/velocity/attitude tolerance.\\"\\"\\"\\n\\n    type: Literal[\\"docked\\"] = \\"docked\\"\\n    final_secondary: SatelliteState\\n    metrics: Metrics = Field(default_factory=Metrics)\\n\\n\\nclass FailedEvent(StreamEventBase):\\n    \\"\\"\\"Terminal failure with the reason (fuel, steps, time, diverged, ...).\\"\\"\\"\\n\\n    type: Literal[\\"failed\\"] = \\"failed\\"\\n    failure_reason: FailureReason\\n    error: str | None = None\\n    final_secondary: SatelliteState | None = None\\n    metrics: Metrics = Field(default_factory=Metrics)\\n\\n\\nDockingEvent = Annotated[\\n    TelemetryEvent | HeartbeatEvent | LlmDecisionEvent | AnomalyEvent | DockedEvent | FailedEvent,\\n    Field(discriminator=\\"type\\"),\\n]\\n", "src/app/schemas/__init__.py": "\\"\\"\\"Pydantic schemas for the docking simulator API.\\n\\nThis is the single source of truth for the wire contracts consumed by the\\nfuture frontend. See README.md section \\"Event schema\\" for the documented\\ncontract.\\n\\"\\"\\"\\n\\nfrom app.schemas.docking import (\\n    AnomalyConfig,\\n    AnomalyType,\\n    ControllerType,\\n    DockingPhase,\\n    FailureReason,\\n    Metrics,\\n    OrbitalElements,\\n    SatelliteState,\\n    SessionSummary,\\n    SimulateRequest,\\n    SimulateResponse,\\n    SimulationConfig,\\n    StopReason,\\n    Vector3,\\n)\\nfrom app.schemas.events import (\\n    AnomalyDetail,\\n    AnomalyEvent,\\n    DockedEvent,\\n    DockingEvent,\\n    FailedEvent,\\n    HeartbeatEvent,\\n    LlmDecisionEvent,\\n    LlmSubgoal,\\n    TelemetryEvent,\\n)\\n\\n__all__ = [\\n    \\"AnomalyConfig\\",\\n    \\"AnomalyDetail\\",\\n    \\"AnomalyEvent\\",\\n    \\"AnomalyType\\",\\n    \\"ControllerType\\",\\n    \\"DockedEvent\\",\\n    \\"DockingEvent\\",\\n    \\"DockingPhase\\",\\n    \\"FailedEvent\\",\\n    \\"FailureReason\\",\\n    \\"HeartbeatEvent\\",\\n    \\"LlmDecisionEvent\\",\\n    \\"LlmSubgoal\\",\\n    \\"Metrics\\",\\n    \\"OrbitalElements\\",\\n    \\"SatelliteState\\",\\n    \\"SessionSummary\\",\\n    \\"SimulateRequest\\",\\n    \\"SimulateResponse\\",\\n    \\"SimulationConfig\\",\\n    \\"StopReason\\",\\n    \\"TelemetryEvent\\",\\n    \\"Vector3\\",\\n]\\n", "src/app/sessions/store.py": "\\"\\"\\"Small, process-local session log. Each SSE viewer has its own cursor.\\n\\n# ponytail: sessions live in memory; use a database and worker queue for a public multi-process deployment.\\n\\"\\"\\"\\n\\nimport asyncio\\nimport uuid\\n\\nfrom app.schemas.docking import SessionSummary\\nfrom app.schemas.events import DockingEvent\\n\\n\\nclass Session:\\n    def __init__(self, session_id: str, queue_size: int = 1000):\\n        self.session_id = session_id\\n        self.event_log: list[DockingEvent] = []\\n        self.summary: SessionSummary | None = None\\n        self.finished = asyncio.Event()\\n        self.task: asyncio.Task | None = None\\n        self.stream_pacing_s = 0.0\\n        self.cancel_requested = False\\n        self.dock_requested = False\\n        self.capture: dict = {}\\n        self.request_data: dict | None = None\\n\\n    async def append(self, event):\\n        event.session_id = self.session_id\\n        event.event_id = len(self.event_log) + 1\\n        self.event_log.append(event)\\n        if self.summary:\\n            self.summary.events_emitted = len(self.event_log)\\n\\n\\nclass InMemorySessionStore:\\n    def __init__(self, queue_size=1000):\\n        self._sessions: dict[str, Session] = {}\\n\\n    async def create(self):\\n        session = Session(uuid.uuid4().hex)\\n        # Keep only the latest 32 completed sessions. Active sessions are capped at the API.\\n        completed = [key for key, value in self._sessions.items() if value.finished.is_set()]\\n        for key in completed[:-31]:\\n            del self._sessions[key]\\n        self._sessions[session.session_id] = session\\n        return session\\n\\n    async def get(self, session_id):\\n        return self._sessions.get(session_id)\\n\\n    async def close(self, session_id):\\n        session = self._sessions.get(session_id)\\n        if session:\\n            session.finished.set()\\n", "src/app/sessions/__init__.py": "\\"\\"\\"Session lifecycle behind a swappable store interface.\\"\\"\\"\\n", "src/app/simulation/anomalies.py": "\\"\\"\\"Anomaly injection.\\n\\nEach configured anomaly is applied at its ``step`` inside the simulation loop:\\n\\n  * sensor_dropout   — the reported relative state seen by the LLM and the\\n                       controller is withheld (gap) or corrupted with Gaussian\\n                       noise for ``duration_steps``. The *truth* state still\\n                       propagates physically; only the observed state is wrong.\\n  * velocity_impulse — an unmodeled instantaneous Δv is applied to the truth\\n                       state (clamped to a sane envelope).\\n  * priority_change  — an instruction (abort) is injected into the LLM context\\n                       only; it does not touch physics.\\n\\nThe anomaly manager is structured so a future frontend/eval harness can see the\\nexact same data: every anomaly also produces an ``AnomalyDetail`` that the\\nrunner embeds into the SSE ``anomaly`` event and into the LLM prompt context.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport numpy as np\\n\\nfrom app.schemas.docking import AnomalyConfig, AnomalyType\\nfrom app.schemas.events import AnomalyDetail\\nfrom app.simulation.dynamics import RigidBodyState\\n\\n\\nclass AnomalyState:\\n    \\"\\"\\"Bookkeeping for one injected anomaly.\\"\\"\\"\\n\\n    def __init__(self, config: AnomalyConfig, rng: np.random.Generator) -> None:\\n        self.config = config\\n        self.rng = rng\\n        self.active_remaining_steps = 0\\n        self.fired = False\\n        self.current_detail: AnomalyDetail | None = None\\n        self.since_last_event_steps = 0\\n        self._impulse_applied = False\\n\\n    def tick(self, step: int, truth: RigidBodyState) -> AnomalyDetail | None:\\n        \\"\\"\\"Advance one physics step. Returns a new AnomalyDetail only when the\\n        anomaly fires or when its effect ends (dropout restoration), so the\\n        stream isn\'t spammed with redundant anomaly events.\\"\\"\\"\\n        self.since_last_event_steps += 1\\n        detail: AnomalyDetail | None = None\\n\\n        if not self.fired and step >= self.config.step:\\n            self.fired = True\\n            self.active_remaining_steps = self.config.duration_steps or 0\\n            self.since_last_event_steps = 0\\n            base = {\\n                \\"type\\": self.config.type,\\n                \\"detail\\": self.config.detail,\\n                \\"step\\": step,\\n            }\\n            if self.config.type is AnomalyType.SENSOR_DROPOUT:\\n                detail = AnomalyDetail(\\n                    **base,\\n                    sensor=\\"relative_state\\",\\n                    impulse_mps=None,\\n                    priority_change=None,\\n                )\\n                self.current_detail = detail\\n            elif self.config.type is AnomalyType.VELOCITY_IMPULSE:\\n                detail = AnomalyDetail(\\n                    **base, sensor=None, impulse_mps=self.config.impulse_mps, priority_change=None\\n                )\\n                self.current_detail = detail\\n            elif self.config.type is AnomalyType.PRIORITY_CHANGE:\\n                detail = AnomalyDetail(\\n                    **base,\\n                    sensor=None,\\n                    impulse_mps=None,\\n                    priority_change=self.config.priority_change,\\n                )\\n                self.current_detail = detail\\n        elif self.fired and self.active_remaining_steps > 0:\\n            self.active_remaining_steps -= 1\\n\\n        # Emit a second event when a sensor-dropout window ends (restored).\\n        if (\\n            self.config.type is AnomalyType.SENSOR_DROPOUT\\n            and self.fired\\n            and self.active_remaining_steps == 0\\n            and self.since_last_event_steps >= 2\\n            and detail is None\\n        ):\\n            detail = AnomalyDetail(\\n                type=self.config.type,\\n                detail=f\\"sensor restored after {self.config.duration_steps} steps\\",\\n                step=step,\\n                sensor=\\"relative_state\\",\\n                impulse_mps=None,\\n                priority_change=None,\\n            )\\n            self.since_last_event_steps = 0\\n            self.current_detail = detail\\n\\n        return detail\\n\\n    @property\\n    def active(self) -> bool:\\n        return self.fired and (\\n            self.active_remaining_steps > 0 or self.config.duration_steps is None\\n        )\\n\\n    @property\\n    def observing_dropout(self) -> bool:\\n        return self.config.type is AnomalyType.SENSOR_DROPOUT and self.active\\n\\n\\nclass AnomalyManager:\\n    \\"\\"\\"Applies configured anomalies to the truth state and produces the\\n    observed state that the LLM + controller act on.\\"\\"\\"\\n\\n    def __init__(self, configs: list[AnomalyConfig], rng: np.random.Generator) -> None:\\n        self.states = [AnomalyState(c, rng) for c in configs]\\n        self.rng = rng\\n        # Noise scale used to corrupt sensor readings (fraction of signal).\\n        self.corruption_fraction = 0.05\\n\\n    def advance(\\n        self, step: int, truth: RigidBodyState\\n    ) -> tuple[RigidBodyState, list[AnomalyDetail]]:\\n        \\"\\"\\"Advance anomaly timelines; return (observed_state, new_anomaly_details).\\"\\"\\"\\n        observed = truth.deep_copy()\\n        details: list[AnomalyDetail] = []\\n\\n        for st in self.states:\\n            evt = st.tick(step, truth)\\n            if evt is not None:\\n                details.append(evt)\\n\\n            if st.observing_dropout:\\n                if st.config.corrupt:\\n                    self._corrupt_observed(observed)\\n                else:\\n                    self._withhold_observed(observed)\\n\\n            if (\\n                st.config.type is AnomalyType.VELOCITY_IMPULSE\\n                and st.fired\\n                and st.active_remaining_steps >= 0\\n            ):\\n                # Apply impulses on the step they fire and while active (once).\\n                if getattr(st, \\"_impulse_applied\\", False) is False:\\n                    impulse = np.asarray(st.config.impulse_mps, dtype=float)\\n                    truth.velocity = truth.velocity + impulse\\n                    observed.velocity = observed.velocity + impulse\\n                    st._impulse_applied = True\\n\\n        if not any(st.observing_dropout for st in self.states):\\n            self._last_known_good = truth.deep_copy()\\n\\n        return observed, details\\n\\n    def priority_change_active(self, step: int) -> str | None:\\n        \\"\\"\\"If a priority-change/abort instruction is active at this step, return it.\\"\\"\\"\\n        for st in self.states:\\n            if (\\n                st.config.type is AnomalyType.PRIORITY_CHANGE\\n                and st.fired\\n                and (st.active_remaining_steps > 0 or st.config.duration_steps is None)\\n            ):\\n                return st.config.priority_change or \\"abort\\"\\n        return None\\n\\n    # ------------------------------------------------------------------\\n\\n    def _corrupt_observed(self, observed: RigidBodyState) -> None:\\n        \\"\\"\\"Add Gaussian noise to the reported relative state.\\"\\"\\"\\n        noise_scale = np.array([10.0, 10.0, 10.0]) * self.corruption_fraction\\n        observed.position = observed.position + self.rng.normal(0.0, noise_scale, 3)\\n        observed.velocity = observed.velocity + self.rng.normal(0.0, 0.05, 3)\\n\\n    def _withhold_observed(self, observed: RigidBodyState) -> None:\\n        \\"\\"\\"Withhold the reading: report the last-known-good state (frozen).\\"\\"\\"\\n        # The runner freezes observed state before propagation decisions anyway;\\n        # here we simply mark it as stale by holding the previous value.\\n        # Since `observed` is a deep-copy of `truth`, the freeze is applied by\\n        # the runner copying the pre-dropout snapshot into it. We do that here\\n        # by keeping a private last-valid copy.\\n\\n        if getattr(self, \\"_last_known_good\\", None) is None:\\n            self._last_known_good = observed.deep_copy()\\n        else:\\n            # \'observed\' currently equals truth; overwrite with the frozen copy.\\n            frozen = self._last_known_good\\n            observed.position = frozen.position.copy()\\n            observed.velocity = frozen.velocity.copy()\\n            observed.quat = frozen.quat.copy()\\n            observed.omega = frozen.omega.copy()\\n\\n    def snapshot_last_known_good(self, state: RigidBodyState) -> None:\\n        \\"\\"\\"Called by the runner once at dropout start — captures the pre-dropout state.\\"\\"\\"\\n        if not hasattr(self, \\"_last_known_good\\"):\\n            self._last_known_good = state.deep_copy()\\n\\n    @property\\n    def total_fired(self) -> int:\\n        return sum(1 for st in self.states if st.fired)\\n\\n    def active_detail(self) -> AnomalyDetail | None:\\n        for st in self.states:\\n            if st.fired and st.current_detail is not None:\\n                return st.current_detail\\n        return None\\n", "src/app/simulation/dynamics.py": "\\"\\"\\"Attitude + relative orbital dynamics.\\n\\nPropagation model\\n-----------------\\nRelative (Hill/RSW frame, origin at the target/primary satellite):\\n\\n    x\'\' - 2n y\' - 3n² x = a_x          (radial)\\n    y\'\' + 2n x\'        = a_y          (along-track)\\n    z\'\' + n² z         = a_z          (cross-track)\\n\\nwith n = sqrt(mu / a³) the mean motion of the primary\'s circular orbit.\\nNumerical integration is semi-implicit symplectic Euler, which preserves the\\nCW drift/oscillation structure far better than explicit Euler over many\\nsteps for this linear system.\\n\\nAttitude uses quaternion kinematics (no Euler angles):\\n\\n    q\' = ½ Ω(ω) q\\n    I ω\' + ω × (Iω) = τ\\n\\nquaternion convention throughout: [w, x, y, z] with w the scalar part.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport math\\n\\nimport numpy as np\\nfrom pydantic import BaseModel, Field\\n\\n# Standard gravitational parameter of Earth [m³/s²].\\nMU_EARTH_M3_S2 = 3.986004418e14\\n\\n# Default principal moments of inertia for a generic ~500 kg chaser [kg·m²].\\nDEFAULT_INERTIA = np.array([600.0, 900.0, 900.0])\\n\\n# Max gyroscopic rotation per rotational sub-step [rad]. Explicit Euler on the\\n# ω×(Iω) term amplifies the transverse spin rate by ~√(1+(ω·dt)²) per step and\\n# compounds to exponential blow-up for large ω·dt; sub-stepping keeps each\\n# rotation small enough that Euler is accurate.\\n_GYRO_SUBSTEP_RAD = 0.05\\n\\n\\ndef mean_motion_radps(semi_major_axis_m: float, mu: float = MU_EARTH_M3_S2) -> float:\\n    \\"\\"\\"Mean motion n of a circular orbit: n = sqrt(mu / a³).\\"\\"\\"\\n    return float(math.sqrt(mu / (semi_major_axis_m**3)))\\n\\n\\n# ---------------------------------------------------------------------------\\n# Quaternion algebra (numpy, [w,x,y,z])\\n# ---------------------------------------------------------------------------\\n\\n\\ndef quat_normalize(q: np.ndarray) -> np.ndarray:\\n    n = np.linalg.norm(q)\\n    return q / n if n > 0.0 else np.array([1.0, 0.0, 0.0, 0.0], dtype=float)\\n\\n\\ndef quat_conjugate(q: np.ndarray) -> np.ndarray:\\n    return np.array([q[0], -q[1], -q[2], -q[3]])\\n\\n\\ndef quat_multiply(p: np.ndarray, q: np.ndarray) -> np.ndarray:\\n    \\"\\"\\"Hamilton product p ⊗ q (both [w,x,y,z]).\\"\\"\\"\\n    pw, px, py, pz = p\\n    qw, qx, qy, qz = q\\n    return np.array(\\n        [\\n            pw * qw - px * qx - py * qy - pz * qz,\\n            pw * qx + px * qw + py * qz - pz * qy,\\n            pw * qy - px * qz + py * qw + pz * qx,\\n            pw * qz + px * qy - py * qx + pz * qw,\\n        ]\\n    )\\n\\n\\ndef quat_rotate(q: np.ndarray, v: np.ndarray) -> np.ndarray:\\n    \\"\\"\\"Rotate a 3-vector by unit quaternion q ([w,x,y,z]).\\"\\"\\"\\n    qv = np.array([0.0, v[0], v[1], v[2]])\\n    q_inv = quat_conjugate(q)\\n    return quat_multiply(quat_multiply(q, qv), q_inv)[1:]\\n\\n\\ndef quat_from_axis_angle(axis: np.ndarray, angle: float) -> np.ndarray:\\n    axis = np.asarray(axis, dtype=float)\\n    norm = np.linalg.norm(axis)\\n    if norm < 1e-12 or abs(angle) < 1e-12:\\n        return np.array([1.0, 0.0, 0.0, 0.0])\\n    axis = axis / norm\\n    half = angle / 2.0\\n    return np.array([math.cos(half), *(axis * math.sin(half))])\\n\\n\\ndef quat_to_rotation_matrix(q: np.ndarray) -> np.ndarray:\\n    \\"\\"\\"Rotation matrix (body->inertial) from unit quaternion [w,x,y,z].\\"\\"\\"\\n    w, x, y, z = q\\n    return np.array(\\n        [\\n            [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],\\n            [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],\\n            [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)],\\n        ]\\n    )\\n\\n\\ndef attitude_error_deg(q1: np.ndarray, q2: np.ndarray) -> float:\\n    \\"\\"\\"Angle between two attitudes in degrees.\\"\\"\\"\\n    dq = quat_multiply(quat_conjugate(quat_normalize(q1)), quat_normalize(q2))\\n    return float(2.0 * math.degrees(math.acos(min(1.0, abs(dq[0])))))\\n\\n\\n# ---------------------------------------------------------------------------\\n# Phase classification\\n# ---------------------------------------------------------------------------\\n\\n\\ndef classify_phase(range_m: float) -> str:\\n    \\"\\"\\"Classify docking phase by relative range (see README).\\"\\"\\"\\n    if range_m <= 0.0:\\n        return \\"docked\\"\\n    if range_m <= 5.0:\\n        return \\"proximity\\"\\n    if range_m <= 30.0:\\n        return \\"final_approach\\"\\n    if range_m <= 1000.0:\\n        return \\"closing\\"\\n    return \\"far_rendezvous\\"\\n\\n\\n# ---------------------------------------------------------------------------\\n# Physics state + integration\\n# ---------------------------------------------------------------------------\\n\\n\\nclass RigidBodyState(BaseModel):\\n    \\"\\"\\"Full continuous state of one satellite (Hill relative for the chaser).\\"\\"\\"\\n\\n    position: np.ndarray = Field(default_factory=lambda: np.zeros(3))\\n    velocity: np.ndarray = Field(default_factory=lambda: np.zeros(3))\\n    quat: np.ndarray = Field(default_factory=lambda: np.array([1.0, 0.0, 0.0, 0.0]))\\n    omega: np.ndarray = Field(default_factory=lambda: np.zeros(3))\\n    fuel_pct: float = Field(default=100.0)\\n    mass_kg: float = Field(default=500.0)\\n    inertia: np.ndarray = Field(default_factory=lambda: DEFAULT_INERTIA.copy())\\n\\n    model_config = {\\"arbitrary_types_allowed\\": True}\\n\\n    def deep_copy(self) -> RigidBodyState:\\n        \\"\\"\\"Return an independent copy (numpy arrays not shared).\\"\\"\\"\\n        return RigidBodyState(\\n            position=self.position.copy(),\\n            velocity=self.velocity.copy(),\\n            quat=self.quat.copy(),\\n            omega=self.omega.copy(),\\n            fuel_pct=self.fuel_pct,\\n            mass_kg=self.mass_kg,\\n            inertia=self.inertia.copy(),\\n        )\\n\\n\\nclass Command(BaseModel):\\n    \\"\\"\\"Thruster command computed by a controller (see controllers/base.py).\\"\\"\\"\\n\\n    force_n: np.ndarray = Field(default_factory=lambda: np.zeros(3))\\n    torque_nm: np.ndarray = Field(default_factory=lambda: np.zeros(3))\\n\\n    model_config = {\\"arbitrary_types_allowed\\": True}\\n\\n\\ndef propagate_chaser(\\n    state: RigidBodyState,\\n    dt: float,\\n    n: float,\\n    command: Command | None = None,\\n) -> RigidBodyState:\\n    \\"\\"\\"Propagate the chaser one fixed step via semi-implicit Euler.\\n\\n    Translational: CW equations with control acceleration a = F/m.\\n    Rotational: q\' = ½Ωq ; I ω\' + ω×(Iω) = τ (Euler step on ω, exact quat step).\\n    \\"\\"\\"\\n    cmd = command or Command()\\n    force = np.asarray(cmd.force_n, dtype=float)\\n    torque = np.asarray(cmd.torque_nm, dtype=float)\\n    m = float(state.mass_kg)\\n\\n    a = force / m\\n\\n    # ---- translational: semi-implicit Euler on the CW linear system ----\\n    x, y, z = state.position\\n    vx, vy, vz = state.velocity\\n\\n    ax_cw = 2.0 * n * vy + 3.0 * n * n * x\\n    ay_cw = -2.0 * n * vx\\n    az_cw = -n * n * z\\n\\n    new_vx = vx + (ax_cw + a[0]) * dt\\n    new_vy = vy + (ay_cw + a[1]) * dt\\n    new_vz = vz + (az_cw + a[2]) * dt\\n\\n    new_x = x + new_vx * dt\\n    new_y = y + new_vy * dt\\n    new_z = z + new_vz * dt\\n\\n    # ---- rotational: quaternion kinematics + rigid-body torque ----\\n    omega = state.omega.copy()\\n    inertia = np.asarray(state.inertia, dtype=float)\\n    # I ω\' = τ - ω × (Iω), integrated with explicit Euler sub-steps sized so\\n    # the gyroscopic rotation per sub-step stays ≲ _GYRO_SUBSTEP_RAD. Without\\n    # this, a fast-spinning chaser (e.g. from a bad policy) goes numerically\\n    # unstable: the ω×(Iω) term amplifies transverse spin exponentially.\\n    omega_norm_0 = float(np.linalg.norm(omega))\\n    n_sub = 1\\n    if np.isfinite(omega_norm_0) and omega_norm_0 > 1e-12:\\n        n_sub = max(1, int(math.ceil(omega_norm_0 * dt / _GYRO_SUBSTEP_RAD)))\\n    h = dt / n_sub\\n    for _ in range(n_sub):\\n        angular_accel = (torque - np.cross(omega, inertia * omega)) / inertia\\n        if not np.all(np.isfinite(angular_accel)):\\n            break\\n        omega = omega + angular_accel * h\\n\\n    new_omega = omega\\n\\n    # Exact quaternion step for constant ω: q_{k+1} = q_k ⊗ exp(½ ω dt)\\n    omega_norm = float(np.linalg.norm(new_omega))\\n    if np.isfinite(omega_norm) and omega_norm > 1e-12:\\n        half_angle = 0.5 * omega_norm * dt\\n        dq = np.array([math.cos(half_angle), *(new_omega / omega_norm * math.sin(half_angle))])\\n        new_quat = quat_normalize(quat_multiply(state.quat, dq))\\n    else:\\n        new_quat = quat_normalize(state.quat)\\n\\n    return RigidBodyState(\\n        position=np.array([new_x, new_y, new_z]),\\n        velocity=np.array([new_vx, new_vy, new_vz]),\\n        quat=new_quat,\\n        omega=new_omega,\\n        fuel_pct=state.fuel_pct,\\n        mass_kg=m,\\n        inertia=inertia,\\n    )\\n", "src/app/simulation/runner.py": "\\"\\"\\"Simulation runner: the loop that drives one docking session.\\n\\nThe runner owns the per-session execution flow:\\n\\n  * converts the request\'s ``SatelliteState`` into the internal physics state,\\n  * derives the mean motion ``n`` from the orbital elements,\\n  * advances the anomaly manager (dropout / impulse / priority change),\\n  * calls the reasoning layer every ``llm_cadence_steps`` for a new sub-goal,\\n  * computes the low-level controller command and propagates the dynamics,\\n  * accumulates evaluation metrics (Δv, ISE, peak thrust, LLM call counts),\\n  * emits the SSE event stream and terminates on docked / failure conditions.\\n\\nAll events are appended to one ordered log; each SSE client reads its own cursor.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport asyncio\\nimport logging\\nfrom typing import Any\\n\\nimport numpy as np\\n\\nfrom app.config import Settings\\nfrom app.llm.reasoning_agent import (\\n    HeuristicAgent,\\n    ReasoningAgent,\\n    ReasoningAgentFactory,\\n    ReasoningInput,\\n    with_retry_heuristic_fallback,\\n)\\nfrom app.llm.rl_advisor import RlAdvisor\\nfrom app.llm.tools import docking_tools\\nfrom app.schemas.docking import (\\n    DockingPhase,\\n    FailureReason,\\n    Metrics,\\n    MotionLimits,\\n    SatelliteState,\\n    SessionSummary,\\n    SimulateRequest,\\n    StopReason,\\n)\\nfrom app.schemas.events import (\\n    AnomalyEvent,\\n    DockedEvent,\\n    FailedEvent,\\n    LlmDecisionEvent,\\n    LlmSubgoal,\\n    PrimaryFrame,\\n    RlAdvisorSuggestion,\\n    TelemetryEvent,\\n)\\nfrom app.sessions.store import Session\\nfrom app.simulation.anomalies import AnomalyManager\\nfrom app.simulation.controllers import ControlContext, build_controller\\nfrom app.simulation.dynamics import (\\n    DEFAULT_INERTIA,\\n    RigidBodyState,\\n    classify_phase,\\n    mean_motion_radps,\\n    propagate_chaser,\\n)\\nfrom app.simulation.safety import capture_checks, enforce_command, limit_velocity\\n\\nlogger = logging.getLogger(\\"llm_docking.runner\\")\\n\\n# Reference attitude: the docking port frame (body +X aligned with port).\\n\\n# Static sub-goal used when the LLM layer is disabled: direct homing.\\n_STATIC_SUBGOAL = LlmSubgoal(\\n    target_position_m=[0.0, 0.0, 0.0],\\n    target_velocity_mps=[0.0, 0.0, 0.0],\\n    closing_speed_mps=0.0,\\n    rationale=\\"LLM disabled: direct homing to the docking port.\\",\\n)\\n\\n\\nclass SimulationRunner:\\n    \\"\\"\\"Runs one simulation session to completion and writes its summary.\\"\\"\\"\\n\\n    def __init__(\\n        self,\\n        session: Session,\\n        request: SimulateRequest,\\n        settings: Settings | None = None,\\n        agent: ReasoningAgent | None = None,\\n        heuristic: HeuristicAgent | None = None,\\n    ) -> None:\\n        self.session = session\\n        self.request = request\\n        self.settings = settings or _default_settings()\\n        self.agent = agent or ReasoningAgentFactory.build(\\n            request.config.provider, model=request.config.model\\n        )\\n        self.heuristic = heuristic or ReasoningAgentFactory.heuristic()\\n\\n        self._metrics = Metrics()\\n        self._last_phase: DockingPhase | None = None\\n        self._last_source: str | None = None\\n        self._last_truth: RigidBodyState | None = None\\n        self._current_step = 0\\n        self._current_time = 0.0\\n        # Primary frame anchors every telemetry event (frontend camera).\\n        self._primary_frame = PrimaryFrame(\\n            position_m=request.primary.position_m, orientation_quat=request.primary.orientation_quat\\n        )\\n        self.session.request_data = request.model_dump(mode=\\"json\\")\\n        self._command_limited = False\\n        self._tool_results = []\\n        self._safety_notes = []\\n        # Lazy RL advisor (hybrid mode: the LLM planner sees a suggestion from\\n        # the trained policy). None = not requested or unavailable.\\n        self._rl_advisor: RlAdvisor | None = None\\n        self._rl_advisor_unavailable = False\\n\\n    # ------------------------------------------------------------------\\n    # Public entry point\\n    # ------------------------------------------------------------------\\n\\n    async def run(self) -> None:\\n        \\"\\"\\"Drive the simulation to a terminal event and finalize the summary.\\"\\"\\"\\n        try:\\n            self._build_initial_summary()\\n\\n            await self._run_loop()\\n        except Exception as exc:  # noqa: BLE001 - degrade to a failed summary\\n            logger.exception(\\"simulation_crashed\\", extra={\\"session_id\\": self.session.session_id})\\n            await self._fail(\\n                step=self._current_step,\\n                sim_time=self._current_time,\\n                reason=FailureReason.UNRECOVERABLE_ANOMALY,\\n                error=str(exc),\\n            )\\n        finally:\\n            self.session.finished.set()\\n\\n    # ------------------------------------------------------------------\\n    # Main loop\\n    # ------------------------------------------------------------------\\n\\n    async def _run_loop(self) -> None:\\n        cfg = self.request.config\\n        dt = cfg.step_interval_s\\n\\n        orbital = self.request.primary.orbital_elements or self.request.secondary.orbital_elements\\n        assert orbital is not None  # guaranteed by SimulateRequest validation\\n        n = orbital.mean_motion_radps or mean_motion_radps(orbital.semi_major_axis_m)\\n\\n        truth = self._to_rigid_body(self.request.secondary)\\n        primary = self.request.primary\\n        truth.position -= np.asarray(primary.position_m)\\n        truth.velocity -= np.asarray(primary.velocity_mps)\\n        from app.simulation.dynamics import quat_conjugate, quat_multiply\\n\\n        truth.quat = quat_multiply(quat_conjugate(np.asarray(primary.orientation_quat)), truth.quat)\\n        truth.omega -= np.asarray(primary.angular_velocity_radps)\\n        rng = np.random.default_rng(cfg.seed)\\n        controller = build_controller(cfg.controller, truth.mass_kg, truth.inertia, n)\\n        ctx = ControlContext()\\n        anomalies = AnomalyManager(cfg.anomalies, rng)\\n\\n        subgoal: LlmSubgoal | None = None\\n        step = 0\\n        sim_time = 0.0\\n        self._current_step = step\\n        self._current_time = sim_time\\n\\n        # Initial state snapshot before any control.\\n        await self._emit_telemetry(truth, step, sim_time, None, self._last_source)\\n\\n        while True:\\n            self._last_truth = truth.deep_copy()\\n            self.session.capture = self._capture(truth)\\n            if self.session.cancel_requested:\\n                await self._fail(step, sim_time, FailureReason.CANCELLED)\\n                return\\n            # ---- advance anomaly timelines (fires configured anomalies) ----\\n            observed, new_details = anomalies.advance(step, truth)\\n            for detail in new_details:\\n                await self._emit(AnomalyEvent(anomaly=detail, step=step, sim_time_s=sim_time))\\n                self._metrics.anomaly_events_count += 1\\n\\n            aborting = anomalies.priority_change_active(step) == \\"abort\\"\\n            sensor_stale = any(s.observing_dropout for s in anomalies.states)\\n            if (\\n                not aborting\\n                and not sensor_stale\\n                and self.session.capture[\\"eligible\\"]\\n                and (cfg.auto_dock or self.session.dock_requested)\\n            ):\\n                await self._docked(truth, step, sim_time)\\n                return\\n\\n            # ---- Plan from a fresh snapshot on cadence or after a fault ----\\n            if cfg.llm_enabled and (step % cfg.llm_cadence_steps == 0 or new_details):\\n                subgoal, source, rl_sugg = await self._plan(\\n                    observed, step, sim_time, anomalies, subgoal\\n                )\\n                self._metrics.total_llm_calls += 1\\n                if source == \\"heuristic_fallback\\":\\n                    self._metrics.llm_fallback_calls += 1\\n                self._last_source = source\\n                controller.reset(ctx)\\n                await self._emit(\\n                    LlmDecisionEvent(\\n                        step=step,\\n                        sim_time_s=sim_time,\\n                        phase=self._phase_of(observed),\\n                        subgoal=subgoal,\\n                        source=source,\\n                        anomaly_context=[\\n                            d\\n                            for d in (st.current_detail for st in anomalies.states)\\n                            if d is not None\\n                        ],\\n                        priority_change=anomalies.priority_change_active(step),\\n                        metrics_so_far=self._metrics.model_copy(deep=True),\\n                        rl_advisor=rl_sugg,\\n                        tool_results=self._tool_results,\\n                        safety_notes=self._safety_notes,\\n                    )\\n                )\\n                if self.session.cancel_requested:\\n                    await self._fail(step, sim_time, FailureReason.CANCELLED)\\n                    return\\n            elif not cfg.llm_enabled and subgoal is None:\\n                subgoal = _STATIC_SUBGOAL\\n                self._last_source = None\\n                controller.reset(ctx)\\n\\n            # ---- low-level control from the *observed* state ----\\n            command = controller.compute(observed, subgoal, dt, ctx)\\n            command, self._command_limited = enforce_command(\\n                truth,\\n                command,\\n                dt,\\n                n,\\n                self.request.primary.velocity_mps,\\n                self.request.secondary.motion_limits,\\n            )\\n\\n            # ---- metrics on truth (physical state), pre-propagation ----\\n            self._metrics.ise += float(np.dot(truth.position, truth.position)) * dt\\n            self._metrics.delta_v_mps += float(np.linalg.norm(command.force_n)) / truth.mass_kg * dt\\n            self._metrics.max_commanded_thrust_n = max(\\n                self._metrics.max_commanded_thrust_n, float(np.linalg.norm(command.force_n))\\n            )\\n\\n            # ---- divergence guards BEFORE propagating: an unstable command\\n            #      (e.g. from a bad RL policy) must never push the state into\\n            #      numerical blow-up — fail the run cleanly instead ----\\n            if (\\n                cfg.max_range_m is not None\\n                and float(np.linalg.norm(truth.position)) > cfg.max_range_m\\n            ) or float(np.linalg.norm(truth.omega)) > cfg.max_omega_radps:\\n                await self._fail(step, sim_time, FailureReason.DIVERGED)\\n                return\\n\\n            # ---- propagate truth, burn fuel ----\\n            truth = propagate_chaser(truth, dt, n, command)\\n            if np.linalg.norm(command.force_n) > 1e-6:\\n                truth.fuel_pct = max(0.0, truth.fuel_pct - cfg.fuel_consumption_rate_pct_per_s * dt)\\n\\n            step += 1\\n            sim_time = step * dt\\n            self._current_step = step\\n            self._current_time = sim_time\\n\\n            # The cooperative primary follows its configured constant angular rate.\\n            from app.simulation.dynamics import quat_from_axis_angle, quat_multiply\\n\\n            primary_rate = np.asarray(self.request.primary.angular_velocity_radps)\\n            rate = float(np.linalg.norm(primary_rate))\\n            if rate > 1e-12:\\n                rotation = quat_from_axis_angle(primary_rate / rate, rate * sim_time)\\n                self._primary_frame.orientation_quat = quat_multiply(\\n                    np.asarray(self.request.primary.orientation_quat), rotation\\n                ).tolist()\\n\\n            self._last_truth = truth.deep_copy()\\n            self._primary_frame.position_m = (\\n                np.asarray(self.request.primary.position_m)\\n                + np.asarray(self.request.primary.velocity_mps) * sim_time\\n            ).tolist()\\n            self.session.capture = self._capture(truth)\\n            absolute_velocity = truth.velocity + np.asarray(self.request.primary.velocity_mps)\\n            if not np.all(np.isfinite(truth.position)) or not np.all(np.isfinite(truth.velocity)):\\n                await self._fail(step, sim_time, FailureReason.DIVERGED)\\n                return\\n            if not np.allclose(\\n                absolute_velocity,\\n                limit_velocity(absolute_velocity, self.request.secondary.motion_limits),\\n                atol=1e-6,\\n            ):\\n                await self._fail(step, sim_time, FailureReason.LIMIT_EXCEEDED)\\n                return\\n            await self._emit_telemetry(truth, step, sim_time, command, self._last_source)\\n            # ---- terminal conditions ----\\n            if (\\n                not aborting\\n                and not sensor_stale\\n                and self._is_docked(truth)\\n                and (cfg.auto_dock or self.session.dock_requested)\\n            ):\\n                await self._docked(truth, step, sim_time)\\n                return\\n            if truth.fuel_pct <= 0.0:\\n                await self._fail(step, sim_time, FailureReason.FUEL_EXHAUSTED)\\n                return\\n            if cfg.max_time_s is not None and sim_time >= cfg.max_time_s:\\n                await self._fail(step, sim_time, FailureReason.MAX_TIME_EXCEEDED)\\n                return\\n            if step >= cfg.max_steps:\\n                await self._fail(step, sim_time, FailureReason.MAX_STEPS_EXCEEDED)\\n                return\\n\\n            await asyncio.sleep(cfg.stream_pacing_s)\\n\\n    # ------------------------------------------------------------------\\n    # Planning\\n    # ------------------------------------------------------------------\\n\\n    async def _plan(\\n        self,\\n        observed: RigidBodyState,\\n        step: int,\\n        sim_time: float,\\n        anomalies: AnomalyManager,\\n        current_subgoal: LlmSubgoal | None,\\n    ) -> tuple[LlmSubgoal, str, RlAdvisorSuggestion | None]:\\n        range_m = float(np.linalg.norm(observed.position))\\n        anomaly_details = [\\n            d for d in (st.current_detail for st in anomalies.states) if d is not None\\n        ]\\n        # Hybrid mode: ask the trained RL policy for an advisory sub-goal.\\n        rl_sugg = self._advisor_suggestion(observed, current_subgoal)\\n        inp = ReasoningInput(\\n            step=step,\\n            sim_time_s=sim_time,\\n            phase=classify_phase(range_m),\\n            range_m=range_m,\\n            position_m=[float(v) for v in observed.position],\\n            velocity_mps=[float(v) for v in observed.velocity],\\n            orientation_quat=[float(v) for v in observed.quat],\\n            angular_velocity_radps=[float(v) for v in observed.omega],\\n            fuel_remaining_pct=observed.fuel_pct,\\n            controller=self.request.config.controller.value,\\n            anomaly_context=anomaly_details,\\n            priority_change=anomalies.priority_change_active(step),\\n            rl_advisor=rl_sugg,\\n            capture=self._capture(observed),\\n            primary_limits=self.request.primary.motion_limits.model_dump(),\\n            secondary_limits=self.request.secondary.motion_limits.model_dump(),\\n        )\\n        self._tool_results = [\\n            {\\"name\\": t.name, \\"result\\": t.invoke({})} for t in docking_tools(inp)[:4]\\n        ]\\n        if isinstance(self.agent, HeuristicAgent):\\n            subgoal, source = await self.agent.reason(inp)\\n        else:\\n            subgoal, source = await with_retry_heuristic_fallback(\\n                self.agent,\\n                inp,\\n                self.heuristic,\\n                max_retries=self.settings.llm_max_retries,\\n                backoff_s=self.settings.llm_retry_backoff_s,\\n            )\\n            self._tool_results = self.agent.last_tools or self._tool_results\\n        self._safety_notes = []\\n        target = np.asarray(subgoal.target_position_m)\\n        if not np.isfinite(target).all() or np.linalg.norm(target) > max(range_m * 1.1, 1.25):\\n            subgoal = self.heuristic.plan(inp)\\n            source = \\"heuristic_fallback\\"\\n            self._safety_notes.append(\\"Rejected non-finite or diverging target.\\")\\n        limits = self.request.secondary.motion_limits\\n        relative_limits = MotionLimits(\\n            max_speed_mps=min(\\n                limits.max_speed_mps, self.heuristic.phase_speed_caps.get(inp.phase, 1.0)\\n            ),\\n            max_horizontal_speed_mps=limits.max_horizontal_speed_mps,\\n            max_vertical_speed_mps=limits.max_vertical_speed_mps,\\n        )\\n        desired_velocity = np.asarray(subgoal.target_velocity_mps)\\n        distance_to_goal = float(np.linalg.norm(target - observed.position))\\n        if (\\n            limits.min_speed_mps > 0\\n            and range_m > 5.0\\n            and inp.priority_change != \\"abort\\"\\n            and not anomaly_details\\n            and np.linalg.norm(desired_velocity) > 1e-9\\n            and np.dot(desired_velocity, observed.position) < 0\\n        ):\\n            # Minimum is a transit preference, bounded by braking distance and phase limits.\\n            transit_speed = min(\\n                limits.min_speed_mps,\\n                np.sqrt(2 * 0.05 * distance_to_goal),\\n                relative_limits.max_speed_mps,\\n            )\\n            if np.linalg.norm(desired_velocity) < transit_speed:\\n                desired_velocity *= transit_speed / np.linalg.norm(desired_velocity)\\n        velocity = limit_velocity(desired_velocity, relative_limits)\\n        if not np.allclose(velocity, subgoal.target_velocity_mps):\\n            self._safety_notes.append(\\n                \\"Target velocity reduced to the configured and phase envelope.\\"\\n            )\\n        subgoal.target_velocity_mps = velocity.tolist()\\n        subgoal.closing_speed_mps = min(subgoal.closing_speed_mps, float(np.linalg.norm(velocity)))\\n        if not subgoal.alternatives:\\n            subgoal.alternatives = [\\n                \\"Faster approach rejected: braking margin decreases.\\",\\n                \\"Immediate capture rejected until all capture checks pass.\\",\\n            ]\\n        return subgoal, source, rl_sugg\\n\\n    def _advisor_suggestion(\\n        self,\\n        observed: RigidBodyState,\\n        current_subgoal: LlmSubgoal | None,\\n    ) -> RlAdvisorSuggestion | None:\\n        cfg = self.request.config\\n        if not cfg.rl_advisor_enabled or self._rl_advisor_unavailable:\\n            return None\\n        if self._rl_advisor is None:\\n            self._rl_advisor = RlAdvisor(\\n                mass_kg=observed.mass_kg,\\n                inertia=observed.inertia,\\n                lookahead_s=cfg.rl_advisor_lookahead_s,\\n            )\\n        baseline = self.heuristic.plan(\\n            ReasoningInput(\\n                step=self._current_step,\\n                sim_time_s=self._current_time,\\n                phase=classify_phase(float(np.linalg.norm(observed.position))),\\n                range_m=float(np.linalg.norm(observed.position)),\\n                position_m=observed.position.tolist(),\\n                velocity_mps=observed.velocity.tolist(),\\n                orientation_quat=observed.quat.tolist(),\\n                angular_velocity_radps=observed.omega.tolist(),\\n                fuel_remaining_pct=observed.fuel_pct,\\n                controller=cfg.controller.value,\\n            )\\n        )\\n        suggestion = self._rl_advisor.suggest(observed, baseline)\\n        if suggestion is None:\\n            self._rl_advisor_unavailable = True  # stop retrying every replan\\n        return suggestion\\n\\n    # ------------------------------------------------------------------\\n    # Terminal events + summary\\n    # ------------------------------------------------------------------\\n\\n    async def _docked(self, truth: RigidBodyState, step: int, sim_time: float) -> None:\\n        final = self._to_satellite(truth)\\n        self._last_phase = DockingPhase.DOCKED\\n        await self._emit(\\n            DockedEvent(\\n                step=step,\\n                sim_time_s=sim_time,\\n                final_secondary=final,\\n                metrics=self._metrics.model_copy(deep=True),\\n            )\\n        )\\n        summary = self.session.summary\\n        assert summary is not None\\n        summary.status = \\"docked\\"\\n        summary.stop_reason = StopReason.DOCKED\\n        summary.failure_reason = None\\n        summary.steps_completed = step\\n        summary.sim_time_s = sim_time\\n        summary.phase = self._last_phase or DockingPhase.DOCKED\\n        summary.final_secondary = final\\n        summary.metrics = self._metrics.model_copy(deep=True)\\n        summary.error = None\\n\\n    async def _fail(\\n        self,\\n        step: int,\\n        sim_time: float,\\n        reason: FailureReason,\\n        error: str | None = None,\\n    ) -> None:\\n        final = self._to_satellite(self._last_truth) if self._last_truth is not None else None\\n        await self._emit(\\n            FailedEvent(\\n                step=step,\\n                sim_time_s=sim_time,\\n                failure_reason=reason,\\n                error=error,\\n                final_secondary=final,\\n                metrics=self._metrics.model_copy(deep=True),\\n            )\\n        )\\n        summary = self.session.summary\\n        assert summary is not None\\n        summary.status = \\"failed\\"\\n        summary.stop_reason = StopReason.FAILED\\n        summary.failure_reason = reason\\n        summary.steps_completed = step\\n        summary.sim_time_s = sim_time\\n        summary.phase = self._last_phase\\n        summary.final_secondary = final\\n        summary.metrics = self._metrics.model_copy(deep=True)\\n        summary.error = error\\n\\n    def _build_initial_summary(self) -> None:\\n        self.session.summary = SessionSummary(\\n            session_id=self.session.session_id,\\n            status=\\"running\\",\\n            controller=self.request.config.controller,\\n            llm_enabled=self.request.config.llm_enabled,\\n            metrics=self._metrics,\\n        )\\n\\n    # ------------------------------------------------------------------\\n    # Helpers\\n    # ------------------------------------------------------------------\\n\\n    async def _emit_telemetry(\\n        self,\\n        state: RigidBodyState,\\n        step: int,\\n        sim_time: float,\\n        command: Any,\\n        llm_source: str | None,\\n    ) -> None:\\n        range_m = float(np.linalg.norm(state.position))\\n        phase = self._phase_of(state)\\n        self._last_phase = phase\\n        self._last_truth = state.deep_copy()\\n        await self._emit(\\n            TelemetryEvent(\\n                step=step,\\n                sim_time_s=sim_time,\\n                phase=phase,\\n                position_m=[float(v) for v in state.position],\\n                velocity_mps=[float(v) for v in state.velocity],\\n                range_m=range_m,\\n                closing_speed_mps=self._closing_speed(state),\\n                fuel_pct=state.fuel_pct,\\n                controller=self.request.config.controller,\\n                llm_source=llm_source,\\n                command_force_n=[float(v) for v in command.force_n]\\n                if command is not None\\n                else None,\\n                command_torque_nm=[float(v) for v in command.torque_nm]\\n                if command is not None\\n                else None,\\n                primary=self._primary_frame.model_copy(deep=True),\\n                metrics_so_far=self._metrics.model_copy(deep=True),\\n                orientation_quat=state.quat.tolist(),\\n                angular_velocity_radps=state.omega.tolist(),\\n                primary_velocity_mps=self.request.primary.velocity_mps,\\n                secondary_velocity_mps=(\\n                    state.velocity + np.asarray(self.request.primary.velocity_mps)\\n                ).tolist(),\\n                capture=self._capture(state),\\n                command_limited=self._command_limited,\\n            )\\n        )\\n        summary = self.session.summary\\n        if summary is not None:\\n            summary.steps_completed = step\\n            summary.sim_time_s = sim_time\\n            summary.phase = phase\\n\\n    async def _emit(self, event: Any) -> None:\\n        await self.session.append(event)\\n\\n    @staticmethod\\n    def _closing_speed(state: RigidBodyState) -> float:\\n        r = state.position\\n        norm = float(np.linalg.norm(r))\\n        if norm < 1e-9:\\n            return 0.0\\n        # Radial closing rate: negative radial velocity, clamped at zero.\\n        return max(0.0, -float(np.dot(state.velocity, r) / norm))\\n\\n    def _capture(self, state):\\n        cfg = self.request.config\\n        return capture_checks(\\n            state,\\n            cfg.dock_position_tol_m,\\n            cfg.dock_velocity_tol_mps,\\n            cfg.dock_attitude_tol_deg,\\n            primary_radius=self.request.primary.capture_radius_m,\\n            secondary_radius=self.request.secondary.capture_radius_m,\\n        )\\n\\n    def _is_docked(self, state):\\n        return self._capture(state)[\\"eligible\\"]\\n\\n    @staticmethod\\n    def _phase_of(state: RigidBodyState) -> DockingPhase:\\n        return DockingPhase(classify_phase(float(np.linalg.norm(state.position))))\\n\\n    @staticmethod\\n    def _to_rigid_body(s: SatelliteState) -> RigidBodyState:\\n        inertia = (\\n            np.asarray(s.inertia_kg_m2, dtype=float)\\n            if s.inertia_kg_m2 is not None\\n            else DEFAULT_INERTIA.copy()\\n        )\\n        return RigidBodyState(\\n            position=np.asarray(s.position_m, dtype=float),\\n            velocity=np.asarray(s.velocity_mps, dtype=float),\\n            quat=np.asarray(s.orientation_quat, dtype=float),\\n            omega=np.asarray(s.angular_velocity_radps, dtype=float),\\n            fuel_pct=s.fuel_remaining_pct,\\n            mass_kg=s.mass_kg,\\n            inertia=inertia,\\n        )\\n\\n    @staticmethod\\n    def _to_satellite(s: RigidBodyState) -> SatelliteState:\\n        return SatelliteState(\\n            position_m=[float(v) for v in s.position],\\n            velocity_mps=[float(v) for v in s.velocity],\\n            orientation_quat=[float(v) for v in s.quat],\\n            angular_velocity_radps=[float(v) for v in s.omega],\\n            fuel_remaining_pct=s.fuel_pct,\\n            mass_kg=s.mass_kg,\\n            inertia_kg_m2=[float(v) for v in s.inertia],\\n        )\\n\\n\\ndef _default_settings() -> Settings:\\n    from app.config import get_settings\\n\\n    return get_settings()\\n", "src/app/simulation/safety.py": "\\"\\"\\"One capture gate and one velocity envelope used by API, tools, and training.\\"\\"\\"\\n\\nimport numpy as np\\n\\nfrom app.schemas.docking import MotionLimits\\nfrom app.simulation.dynamics import Command, attitude_error_deg, propagate_chaser\\n\\nCAPTURE_RANGE_M = 1.25\\n\\n\\ndef limit_velocity(velocity, limits: MotionLimits):\\n    velocity = np.asarray(velocity, dtype=float).copy()\\n    velocity[2] = np.clip(\\n        velocity[2], -limits.max_vertical_speed_mps, limits.max_vertical_speed_mps\\n    )\\n    horizontal = np.linalg.norm(velocity[:2])\\n    if horizontal > limits.max_horizontal_speed_mps:\\n        velocity[:2] *= limits.max_horizontal_speed_mps / horizontal\\n    speed = np.linalg.norm(velocity)\\n    if speed > limits.max_speed_mps:\\n        velocity *= limits.max_speed_mps / speed\\n    return velocity\\n\\n\\ndef capture_checks(\\n    state,\\n    position_tol=1.25,\\n    velocity_tol=0.1,\\n    attitude_tol=2.0,\\n    omega_tol=0.05,\\n    primary_radius=1.25,\\n    secondary_radius=1.25,\\n):\\n    \\"\\"\\"Each port must be inside the other port\'s capture radius.\\"\\"\\"\\n    radius = min(CAPTURE_RANGE_M, position_tol, primary_radius, secondary_radius)\\n    checks = {\\n        \\"range\\": float(np.linalg.norm(state.position)) <= radius,\\n        \\"relative_speed\\": float(np.linalg.norm(state.velocity)) <= velocity_tol,\\n        \\"alignment\\": attitude_error_deg(state.quat, np.array([1.0, 0.0, 0.0, 0.0])) <= attitude_tol,\\n        \\"rotation_rate\\": float(np.linalg.norm(state.omega)) <= omega_tol,\\n    }\\n    if not all(\\n        np.isfinite(values).all()\\n        for values in [state.position, state.velocity, state.quat, state.omega]\\n    ):\\n        checks = {key: False for key in checks}\\n    return {\\"eligible\\": all(checks.values()), \\"checks\\": checks, \\"shared_range_m\\": radius}\\n\\n\\ndef enforce_command(state, command, dt, n, primary_velocity, limits):\\n    \\"\\"\\"Adjust force before propagation; never silently clip the physical state.\\"\\"\\"\\n    primary_velocity = np.asarray(primary_velocity)\\n    force = np.clip(command.force_n, -50.0, 50.0)\\n    torque = np.clip(command.torque_nm, -20.0, 20.0)\\n    drift = (propagate_chaser(state, dt, n, Command()).velocity - state.velocity) / dt\\n    baseline = np.clip(-drift * state.mass_kg, -50.0, 50.0)\\n\\n    def feasible(candidate):\\n        velocity = state.velocity + (drift + candidate / state.mass_kg) * dt + primary_velocity\\n        return np.allclose(velocity, limit_velocity(velocity, limits), atol=1e-9, rtol=0)\\n\\n    # Bisection stays within both the convex velocity envelope and actuator bounds.\\n    if not feasible(force) and feasible(baseline):\\n        low, high = 0.0, 1.0\\n        for _ in range(30):\\n            fraction = (low + high) / 2\\n            candidate = baseline + fraction * (force - baseline)\\n            if feasible(candidate):\\n                low = fraction\\n            else:\\n                high = fraction\\n        force = baseline + low * (force - baseline)\\n    adjusted = not np.allclose(force, command.force_n) or not np.allclose(torque, command.torque_nm)\\n    return Command(force_n=force, torque_nm=torque), adjusted\\n", "src/app/simulation/__init__.py": "\\"\\"\\"Simulation core: dynamics, controllers, phase logic, runner.\\"\\"\\"\\n", "src/app/simulation/controllers/base.py": "\\"\\"\\"Controller interface.\\n\\nEvery low-level controller (PID, LQR, and the future RL/PPO policy) implements\\nthe ``Controller`` protocol below. The simulation loop only ever talks to:\\n\\n    controller.compute(state, subgoal, dt) -> Command\\n\\nNo other subsystem knows which concrete controller is running, so a third\\nimplementation (RL policy) can be dropped into ``app/simulation/controllers/``\\nand registered in the ``build_controller`` factory without touching the loop.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom abc import ABC, abstractmethod\\n\\nimport numpy as np\\n\\nfrom app.schemas.docking import ControllerType\\nfrom app.schemas.events import LlmSubgoal\\nfrom app.simulation.dynamics import Command, RigidBodyState\\n\\n\\nclass ControlContext:\\n    \\"\\"\\"Per-controller bookkeeping a controller may keep between calls.\\n\\n    PID uses it for the integral term; LQR ignores everything except the\\n    gain matrix (which it caches on construction). A future RL controller\\n    would hold its policy weights here.\\n    \\"\\"\\"\\n\\n    def __init__(self) -> None:\\n        # PID integral of position error [m·s], reset per sub-goal.\\n        self.position_integral: np.ndarray = np.zeros(3)\\n        self.last_position_error: np.ndarray | None = None\\n        # Closed-loop gain for LQR (computed once at build time, cached).\\n        self.lqr_gain: np.ndarray | None = None\\n        # LQR continuous-time solver working matrices (cached).\\n        self.lqr_solver_state: tuple | None = None\\n\\n\\nclass Controller(ABC):\\n    \\"\\"\\"Common interface for low-level controllers.\\"\\"\\"\\n\\n    name: ControllerType\\n\\n    def __init__(self, mass_kg: float, inertia: np.ndarray) -> None:\\n        self.mass_kg = float(mass_kg)\\n        self.inertia = np.asarray(inertia, dtype=float).copy()\\n\\n    @abstractmethod\\n    def compute(\\n        self, state: RigidBodyState, subgoal: LlmSubgoal, dt: float, ctx: ControlContext\\n    ) -> Command:\\n        \\"\\"\\"Compute the thruster command to track ``subgoal`` from ``state``.\\"\\"\\"\\n\\n    @abstractmethod\\n    def reset(self, ctx: ControlContext) -> None:\\n        \\"\\"\\"Reset internal state (called when a new LLM sub-goal arrives).\\"\\"\\"\\n\\n\\ndef build_controller(\\n    controller_type: ControllerType,\\n    mass_kg: float,\\n    inertia: np.ndarray,\\n    n: float,\\n) -> Controller:\\n    \\"\\"\\"Factory: maps a ControllerType to a concrete Controller instance.\\n\\n    A future RL/PPO controller adds an ``elif controller_type is ControllerType.RL``\\n    branch here (plus the enum member in the schema). Nothing else changes.\\n    \\"\\"\\"\\n    if isinstance(controller_type, str):\\n        controller_type = ControllerType(controller_type)\\n\\n    if controller_type is ControllerType.PID:\\n        from app.simulation.controllers.pid import PidController\\n\\n        return PidController(mass_kg=mass_kg, inertia=inertia)\\n    if controller_type is ControllerType.LQR:\\n        from app.simulation.controllers.lqr import LqrController\\n\\n        return LqrController(mass_kg=mass_kg, inertia=inertia, n=n)\\n    if controller_type is ControllerType.RL:\\n        from app.simulation.controllers.rl import RlController\\n\\n        return RlController(mass_kg=mass_kg, inertia=inertia)\\n    raise ValueError(f\\"Unknown controller type: {controller_type}\\")\\n", "src/app/simulation/controllers/lqr.py": "\\"\\"\\"LQR low-level controller.\\n\\n    J = ∫₀^∞ (xᵀQx + uᵀRu) dt   =>   u* = -K x\\n\\nThe state is the full 12-DOF stacked vector\\n    x = [delta_r (3), delta_v (3), delta_angle (3), delta_omega (3)]\\nwith errors relative to the LLM sub-goal (position/velocity target). The\\nattitude error uses the rotation-vector (axis·angle) representation, so the\\nwhole controller is a single continuous-time LQR over the linearized CW +\\nquaternion kinematics — no Euler angles.\\n\\nThe algebraic Riccati equation is solved numerically with the scipy port\\n``scipy.linalg.solve_continuous_are`` when scipy is available, otherwise a\\nhand-rolled iterative (Kleinman) solver is used. Only numpy is a hard\\ndependency.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom functools import lru_cache\\n\\nimport numpy as np\\n\\nfrom app.schemas.docking import ControllerType\\nfrom app.schemas.events import LlmSubgoal\\nfrom app.simulation.controllers.base import ControlContext, Controller\\nfrom app.simulation.controllers.lqr_riccati import solve_continuous_are\\nfrom app.simulation.dynamics import Command, RigidBodyState\\n\\n# ---------------------------------------------------------------------------\\n# Tuning. Q penalizes position/velocity errors; R penalizes control effort.\\n# ---------------------------------------------------------------------------\\nQ_TRANSLATION = 1.0  # per m²  (position error)\\nQ_VELOCITY = 0.2  # per (m/s)²\\nQ_ANGLE = 8.0  # per rad² (attitude error)\\nQ_RATE = 0.5  # per (rad/s)²\\nR_FORCE = 1e-3  # per N²\\nR_TORQUE = 0.1  # per (N·m)²\\n\\nMAX_FORCE_N = 50.0\\nMAX_TORQUE_NM = 20.0\\n\\n\\nclass LqrController(Controller):\\n    name = ControllerType.LQR\\n\\n    def __init__(self, mass_kg: float, inertia: np.ndarray, n: float) -> None:\\n        super().__init__(mass_kg=float(mass_kg), inertia=np.asarray(inertia, dtype=float))\\n        self.n = float(n)\\n        self._assembly = self._build_linear_system()\\n\\n    # ------------------------------------------------------------------\\n    # Linear-time-invariant model assembly (CW + q-kinematics, cached)\\n    # ------------------------------------------------------------------\\n\\n    def _build_linear_system(self) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:\\n        \\"\\"\\"Return (A, B, Q, R) for the continuous-time LTI system ẋ = Ax + Bu.\\"\\"\\"\\n        n = self.n\\n        mass = self.mass_kg\\n        Ix, Iy, Iz = self.inertia\\n\\n        # -- Translational: CW — ẋ = A_t x_t + B_t u_t,  x_t = [r; v] --\\n        A_t = np.array(\\n            [\\n                [0.0, 0.0, 0.0, 1.0, 0.0, 0.0],\\n                [0.0, 0.0, 0.0, 0.0, 1.0, 0.0],\\n                [0.0, 0.0, 0.0, 0.0, 0.0, 1.0],\\n                [3.0 * n * n, 0.0, 0.0, 0.0, 2.0 * n, 0.0],\\n                [0.0, 0.0, 0.0, -2.0 * n, 0.0, 0.0],\\n                [0.0, 0.0, -n * n, 0.0, 0.0, 0.0],\\n            ]\\n        )\\n        B_t = np.zeros((6, 3))\\n        B_t[3:, :] = np.eye(3) / mass\\n\\n        # -- Rotational: δθ dynamics from q-kinematics linearized about zero\\n        #    (δθ\' ≈ δω, I δω\' ≈ -ω×(Iω) + τ, linearized δω\' = I⁻¹ τ).\\n        A_r = np.zeros((6, 6))\\n        A_r[0:3, 3:6] = np.eye(3)\\n        B_r = np.vstack((np.zeros((3, 3)), np.diag(1.0 / np.array([Ix, Iy, Iz]))))\\n\\n        # -- Combine: 12-state LTI --\\n        A = np.zeros((12, 12))\\n        A[:6, :6] = A_t\\n        A[6:, 6:] = A_r\\n        B = np.zeros((12, 6))\\n        B[:6, :3] = B_t\\n        B[6:, 3:] = B_r\\n\\n        # -- Cost matrices --\\n        Q = np.zeros((12, 12))\\n        Q[0, 0] = Q[1, 1] = Q[2, 2] = Q_TRANSLATION\\n        Q[3, 3] = Q[4, 4] = Q[5, 5] = Q_VELOCITY\\n        Q[6, 6] = Q[7, 7] = Q[8, 8] = Q_ANGLE\\n        Q[9, 9] = Q[10, 10] = Q[11, 11] = Q_RATE\\n\\n        R = np.zeros((6, 6))\\n        R[0, 0] = R[1, 1] = R[2, 2] = R_FORCE\\n        R[3, 3] = R[4, 4] = R[5, 5] = R_TORQUE\\n\\n        return A, B, Q, R\\n\\n    @property\\n    def gain(self) -> np.ndarray:\\n        \\"\\"\\"Close-loop gain K (6x12) solving the CARE, cached per controller.\\"\\"\\"\\n        key = (self.n, self.mass_kg, *self.inertia)\\n        return self._gain_cached(*key)\\n\\n    @staticmethod\\n    @lru_cache(maxsize=64)\\n    def _gain_cached(n: float, mass: float, ix: float, iy: float, iz: float) -> np.ndarray:\\n        inst = LqrController(mass_kg=mass, inertia=np.array([ix, iy, iz]), n=n)\\n        A, B, Q, R = inst._assembly\\n        P = solve_continuous_are(A, B, Q, R)\\n        return np.linalg.solve(R, B.T @ P)\\n\\n    def compute(\\n        self, state: RigidBodyState, subgoal: LlmSubgoal, dt: float, ctx: ControlContext\\n    ) -> Command:\\n        # -- Translational error state: error = state - target, so the error\\n        #    obeys the same CW dynamics as the state (targets are constant)\\n        #    and the LTI model ẋ = Ax + Bu has consistent signs. Without this\\n        #    convention the closed loop is anti-damped (positive feedback).\\n        r_err = state.position - np.asarray(subgoal.target_position_m, dtype=float)\\n        v_err = state.velocity - np.asarray(subgoal.target_velocity_mps, dtype=float)\\n\\n        # -- Attitude error state: rotation vector of the current attitude\\n        #    relative to the docking-port frame (identity quat), so ψ̇ = ω and\\n        #    the rotational block is a driven double integrator. atan2 keeps\\n        #    180° errors exact and lets the LQR null roll as well as pointing.\\n        w = float(state.quat[0])\\n        v = state.quat[1:]\\n        v_norm = float(np.linalg.norm(v))\\n        if v_norm > 1e-12:\\n            angle_err = (2.0 * np.arctan2(v_norm, w)) * (v / v_norm)\\n        else:\\n            angle_err = np.zeros(3)\\n\\n        omega_err = state.omega  # null body rates\\n\\n        x = np.concatenate([r_err, v_err, angle_err, omega_err])\\n\\n        u = -self.gain @ x\\n        force = np.clip(u[:3], -MAX_FORCE_N, MAX_FORCE_N)\\n        torque = np.clip(u[3:], -MAX_TORQUE_NM, MAX_TORQUE_NM)\\n        return Command(force_n=force, torque_nm=torque)\\n\\n    def reset(self, ctx: ControlContext) -> None:\\n        ctx.position_integral = np.zeros(3)\\n        ctx.last_position_error = None\\n", "src/app/simulation/controllers/lqr_riccati.py": "\\"\\"\\"Use SciPy\'s standard Riccati solver for the LQR gain.\\"\\"\\"\\n\\nfrom scipy.linalg import solve_continuous_are\\n\\n__all__ = [\\"solve_continuous_are\\"]\\n", "src/app/simulation/controllers/pid.py": "\\"\\"\\"PID low-level controller.\\n\\n    u(t) = Kp·e(t) + Ki·∫e dt + Kd·de/dt\\n\\nTracks the LLM sub-goal: position error is relative to ``target_position_m``\\nand velocity error to ``target_velocity_mps``. Two separate PID loops run:\\none for translation (force), one for attitude (torque, tracking a zero-relative\\nrate and a level attitude derived from the closing direction).\\n\\nGains are mass-scaled (spring-damper with ωn ≈ 0.09 rad/s, ζ = 0.9 for a\\ngeneric chaser) and phase-scaled so the controller is gentle at range and\\nfirm close in, which keeps the maneuver stable across the full far -> docked\\nrange while a single fixed gain set would oscillate or stall.\\n\\nThe attitude loop drives the *full* attitude to the docking-port frame\\n(identity quat) with nulled body rates. Translation is commanded in the Hill\\nframe, so attitude alignment and docking are decoupled and reachable from any\\napproach direction.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport math\\n\\nimport numpy as np\\n\\nfrom app.schemas.docking import ControllerType\\nfrom app.schemas.events import LlmSubgoal\\nfrom app.simulation.controllers.base import ControlContext, Controller\\nfrom app.simulation.dynamics import (\\n    Command,\\n    RigidBodyState,\\n    quat_conjugate,\\n    quat_multiply,\\n)\\n\\n# Docking-port frame: Hill-frame aligned (identity quat) with nulled rates.\\n_IDENTITY_QUAT = np.array([1.0, 0.0, 0.0, 0.0])\\n\\n# Translation loop: mass-scaled spring-damper (ωn², damping ratio ζ).\\nPOS_OMEGA2 = 0.008  # ωn ≈ 0.089 rad/s for the nominal 500 kg chaser\\nPOS_ZETA = 0.9\\nKI_POS = 0.002  # integral gain, N per (m·s) of accumulated error\\n\\n# Attitude gains (per rad of error / rad/s of rate error).\\nKP_ATT = 2.0\\nKD_ATT = 1.0\\n\\nMAX_FORCE_N = 50.0  # per-axis thruster saturation [N]\\nMAX_TORQUE_NM = 20.0\\n\\n# Phase-scaling envelope: gain multiplier ramps from `_FAR_SCALE` at range to\\n# 1.0 as the chaser reaches the port (proximity factor below).\\n_FAR_SCALE = 0.35\\n_PROXIMITY_DECAY_M = 25.0\\n\\n\\nclass PidController(Controller):\\n    name = ControllerType.PID\\n\\n    def __init__(self, mass_kg: float, inertia: np.ndarray) -> None:\\n        super().__init__(mass_kg=float(mass_kg), inertia=np.asarray(inertia, dtype=float))\\n        # Mass-scaled proportional/damping gains [N/m] and [N/(m/s)].\\n        self._kp_pos = self.mass_kg * POS_OMEGA2\\n        self._kd_pos = self.mass_kg * 2.0 * POS_ZETA * math.sqrt(POS_OMEGA2)\\n\\n    def compute(\\n        self, state: RigidBodyState, subgoal: LlmSubgoal, dt: float, ctx: ControlContext\\n    ) -> Command:\\n        pos_err = np.asarray(subgoal.target_position_m, dtype=float) - state.position\\n\\n        # Phase-dependent scaling: gentle far, firm close-in.\\n        range_m = float(np.linalg.norm(state.position))\\n        prox = float(math.exp(-range_m / _PROXIMITY_DECAY_M))  # 1 at port -> 0 far away\\n        kp = self._kp_pos * (_FAR_SCALE + (1.0 - _FAR_SCALE) * prox)\\n        kd = self._kd_pos * (_FAR_SCALE + (1.0 - _FAR_SCALE) * prox)\\n        ki = KI_POS * (_FAR_SCALE + (1.0 - _FAR_SCALE) * prox)\\n\\n        # Integral term (reset by the runner via reset() when a new sub-goal arrives).\\n        ctx.position_integral += pos_err * dt\\n        ctx.position_integral = np.clip(ctx.position_integral, -200.0, 200.0)\\n\\n        # Velocity error relative to the sub-goal\'s closing-rate envelope.\\n        target_vel = np.asarray(subgoal.target_velocity_mps, dtype=float)\\n        vel_err = target_vel - state.velocity\\n\\n        force = kp * pos_err + ki * ctx.position_integral + kd * vel_err\\n        force = np.clip(force, -MAX_FORCE_N, MAX_FORCE_N)\\n\\n        # ---- attitude loop: drive the full attitude to the docking-port\\n        #      frame (identity quat), nulling body rates. The rotation vector\\n        #      via atan2 handles 180° errors exactly.\\n        dq = quat_multiply(quat_conjugate(state.quat), _IDENTITY_QUAT)\\n        if dq[0] < 0:\\n            dq = -dq\\n        w = float(dq[0])\\n        v = dq[1:]\\n        v_norm = float(np.linalg.norm(v))\\n        if v_norm > 1e-12:\\n            rot_vec = (2.0 * np.arctan2(v_norm, w)) * (v / v_norm)\\n        else:\\n            rot_vec = np.zeros(3)\\n        torque = state.inertia * (0.008 * rot_vec - 0.18 * state.omega)\\n        torque = np.clip(torque, -MAX_TORQUE_NM, MAX_TORQUE_NM)\\n\\n        return Command(force_n=force, torque_nm=torque)\\n\\n    def reset(self, ctx: ControlContext) -> None:\\n        ctx.position_integral = np.zeros(3)\\n        ctx.last_position_error = None\\n", "src/app/simulation/controllers/rl.py": "\\"\\"\\"RL/PPO low-level controller: wraps a trained Stable-Baselines3 policy.\\n\\nImplements the same ``Controller.compute(state, subgoal, dt, ctx)`` interface\\nas PID and LQR, so the runner treats it identically. The policy was trained\\n(``app/simulation/rl/train_ppo.py``) on the 14-dim observation built by\\n``build_rl_observation`` — relative position/velocity to the current sub-goal,\\nquaternion, angular velocity, fuel fraction — and normalized with the\\n``VecNormalize`` stats saved next to the checkpoint.\\n\\n**Both files must be loaded together.** Feeding unnormalized observations to\\nthe policy silently produces near-random actions, because the network learned\\non normalized inputs. The conformance test in ``tests/test_rl.py`` enforces\\nthat the observation builder used here is byte-identical to the one used in\\ntraining.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport numpy as np\\n\\nfrom app.config import get_settings\\nfrom app.schemas.docking import ControllerType\\nfrom app.schemas.events import LlmSubgoal\\nfrom app.simulation.controllers.base import ControlContext, Controller\\nfrom app.simulation.dynamics import Command, RigidBodyState\\nfrom app.simulation.rl.docking_env import (\\n    MAX_ACCEL_MPS2,\\n    MAX_TORQUE_NM,\\n    DockingEnv,\\n    build_rl_observation,\\n)\\n\\n\\nclass RlController(Controller):\\n    name = ControllerType.RL\\n\\n    def __init__(\\n        self,\\n        mass_kg: float,\\n        inertia: np.ndarray,\\n        model_path: str | None = None,\\n        normalize_path: str | None = None,\\n        device: str = \\"cpu\\",\\n    ) -> None:\\n        super().__init__(mass_kg=float(mass_kg), inertia=np.asarray(inertia, dtype=float))\\n        settings = get_settings()\\n        self.model_path = model_path or settings.rl_model_path\\n        self.normalize_path = normalize_path or settings.rl_normalize_path\\n        if not self.model_path or not self.normalize_path:\\n            raise ValueError(\\n                \\"controller=\'rl\' requires LLM_DOCKING_RL_MODEL_PATH and \\"\\n                \\"LLM_DOCKING_RL_NORMALIZE_PATH (train a policy with \\"\\n                \\"`python -m app.simulation.rl.train_ppo` first)\\"\\n            )\\n\\n        from stable_baselines3 import PPO  # deferred: optional dependency\\n        from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize\\n\\n        self._model = PPO.load(self.model_path, device=device)\\n        # VecNormalize.load re-wraps a venv (only used for its spaces/attrs;\\n        # the running stats come from the saved pickle).\\n        dummy = DummyVecEnv([lambda: DockingEnv(seed=0)])\\n        self._vecnorm = VecNormalize.load(self.normalize_path, dummy)\\n        self._vecnorm.training = False  # freeze the running stats at inference\\n\\n    def compute(\\n        self, state: RigidBodyState, subgoal: LlmSubgoal, dt: float, ctx: ControlContext\\n    ) -> Command:\\n        obs = build_rl_observation(state, subgoal)[np.newaxis, :].astype(np.float32)\\n        obs_norm = self._vecnorm.normalize_obs(obs)\\n        action, _ = self._model.predict(obs_norm, deterministic=True)\\n        a = np.clip(np.asarray(action[0], dtype=float), -1.0, 1.0)\\n\\n        accel = a[:3] * MAX_ACCEL_MPS2\\n        torque = a[3:] * MAX_TORQUE_NM\\n        return Command(force_n=accel * self.mass_kg, torque_nm=torque)\\n\\n    def reset(self, ctx: ControlContext) -> None:\\n        pass  # stateless policy: nothing to reset between sub-goals\\n", "src/app/simulation/controllers/__init__.py": "\\"\\"\\"Low-level controllers. All implement the ``Controller`` protocol.\\n\\nPID, LQR, and the RL/PPO policy controller are interchangeable behind\\n``build_controller`` — the runner never knows which one is running.\\n\\"\\"\\"\\n\\nfrom app.simulation.controllers.base import ControlContext, Controller, build_controller\\nfrom app.simulation.controllers.lqr import LqrController\\nfrom app.simulation.controllers.pid import PidController\\nfrom app.simulation.controllers.rl import RlController\\n\\n__all__ = [\\n    \\"Controller\\",\\n    \\"ControlContext\\",\\n    \\"build_controller\\",\\n    \\"LqrController\\",\\n    \\"PidController\\",\\n    \\"RlController\\",\\n]\\n", "src/app/simulation/rl/docking_env.py": "\\"\\"\\"Gymnasium environment for RL/PPO docking.\\n\\nWraps the **same physics the service runner uses** — ``propagate_chaser``,\\nthe fuel-burn rule, and the exact docked/diverged conditions — so a policy\\ntrained here behaves identically at inference inside ``SimulationRunner``.\\n\\nObservation (14-dim, relative to the current sub-goal):\\n    [rel_pos(3), rel_vel(3), quat(4), ang_vel(3), fuel_frac(1)]\\n\\nThe sub-goal is *not* a fixed origin: at runtime it is whatever the reasoning\\nlayer (LLM or heuristic) last issued, so the policy is trained to drive\\n``(state - subgoal)`` to zero. The sub-goal itself is re-issued periodically\\nby the same ``HeuristicAgent.plan`` the service uses, giving the policy the\\nsame moving-target distribution it will see in production.\\n\\nAction (6-dim continuous in [-1, 1]):\\n    [ax, ay, az, tx, ty, tz]  — scaled internally to real actuator limits\\n    (MAX_ACCEL_MPS2 = 50 N / 500 kg, MAX_TORQUE_NM = 20 N·m).\\n\\nReward (per spec Eq. 9–11):\\n    r = -w1‖Δp‖ - w2‖Δv‖ - w3‖Δq‖ + r_success - w4‖u‖ + potential shaping\\n\\nDesign notes (the failure modes this is built to avoid):\\n\\n* **Divergence guard + penalty.** A random early-training policy can fling the\\n  chaser to extreme positions, which explodes PPO\'s value loss. Episodes\\n  terminate with a large negative penalty when ``‖r‖ > diverged_bound_m``, and\\n  the penalty is larger than the worst accumulated in-episode cost of a\\n  mediocre-but-surviving run — otherwise PPO learns to *fail fast and cheap*\\n  (fly away immediately) rather than learn to dock, producing a clean-looking\\n  improving reward curve while doing the opposite of the task.\\n\\n* **Potential-based shaping.** On top of the absolute error penalty, a shaping\\n  term rewards reducing ‖Δp‖ step-over-step, giving early training a clearer\\n  gradient toward the target.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport math\\n\\nimport gymnasium as gym\\nimport numpy as np\\nfrom gymnasium import spaces\\n\\nfrom app.llm.reasoning_agent import HeuristicAgent, ReasoningInput\\nfrom app.schemas.docking import MotionLimits\\nfrom app.schemas.events import LlmSubgoal\\nfrom app.simulation.dynamics import (\\n    DEFAULT_INERTIA,\\n    Command,\\n    RigidBodyState,\\n    attitude_error_deg,\\n    classify_phase,\\n    propagate_chaser,\\n)\\nfrom app.simulation.safety import capture_checks, enforce_command, limit_velocity\\n\\n# Actuator limits shared with the PID/LQR controllers and the runner.\\nMAX_FORCE_N = 50.0  # per-axis thruster saturation [N]\\nMAX_TORQUE_NM = 20.0  # per-axis torque saturation [N·m]\\nMAX_ACCEL_MPS2 = MAX_FORCE_N / 500.0  # 0.1 m/s² on the nominal 500 kg chaser\\n\\n# Docking-port frame (body +X aligned with port).\\n_IDENTITY_QUAT = np.array([1.0, 0.0, 0.0, 0.0])\\n\\nOBS_DIM = 14\\nACTION_DIM = 6\\n\\nDEFAULT_REWARD_WEIGHTS = {\\"w_pos\\": 1.0, \\"w_vel\\": 1.0, \\"w_att\\": 1.0, \\"w_ctrl\\": 0.1}\\n\\n\\ndef build_rl_observation(state: RigidBodyState, subgoal: LlmSubgoal) -> np.ndarray:\\n    \\"\\"\\"The 14-dim observation, expressed relative to the current sub-goal.\\n\\n    Must be byte-identical between training (env) and inference\\n    (RlController) — a conformance test enforces this.\\n    \\"\\"\\"\\n    dp = np.asarray(state.position, dtype=float) - np.asarray(\\n        subgoal.target_position_m, dtype=float\\n    )\\n    dv = np.asarray(state.velocity, dtype=float) - np.asarray(\\n        subgoal.target_velocity_mps, dtype=float\\n    )\\n    return np.concatenate(\\n        [\\n            dp,\\n            dv,\\n            np.asarray(state.quat, dtype=float),\\n            np.asarray(state.omega, dtype=float),\\n            [state.fuel_pct / 100.0],\\n        ]\\n    ).astype(np.float32)\\n\\n\\nclass DockingEnv(gym.Env):\\n    \\"\\"\\"Relative-position + attitude docking task (Hill/RSW, quaternion).\\"\\"\\"\\n\\n    metadata = {\\"render_modes\\": []}\\n\\n    def __init__(\\n        self,\\n        *,\\n        mean_motion_radps: float = 1.1e-3,\\n        dt: float = 1.0,\\n        max_steps: int = 300,\\n        mass_kg: float = 500.0,\\n        inertia: np.ndarray | None = None,\\n        max_accel_mps2: float = MAX_ACCEL_MPS2,\\n        max_torque_nm: float = MAX_TORQUE_NM,\\n        subgoal_cadence_steps: int = 10,\\n        dock_position_tol_m: float = 1.25,\\n        dock_velocity_tol_mps: float = 0.1,\\n        dock_attitude_tol_deg: float = 2.0,\\n        dock_omega_tol_radps: float = 0.05,\\n        diverged_bound_m: float = 1500.0,\\n        max_omega_radps: float = 5.0,  # tumble guard, mirrors runner max_omega_radps\\n        reward_weights: dict[str, float] | None = None,\\n        success_reward: float = 10.0,\\n        diverged_penalty: float = 50.0,\\n        fuel_out_penalty: float = 10.0,\\n        shaping_scale: float = 1.0,\\n        fuel_consumption_rate_pct_per_s: float = 0.05,\\n        initial_range_m: tuple[float, float] = (20.0, 400.0),\\n        motion_limits: MotionLimits | None = None,\\n        seed: int | None = None,\\n    ) -> None:\\n        super().__init__()\\n        self.n = float(mean_motion_radps)\\n        self.dt = float(dt)\\n        self.max_steps = int(max_steps)\\n        self.mass_kg = float(mass_kg)\\n        self.inertia = np.asarray(inertia if inertia is not None else DEFAULT_INERTIA, dtype=float)\\n        self.max_accel = float(max_accel_mps2)\\n        self.max_torque = float(max_torque_nm)\\n        self.subgoal_cadence_steps = int(subgoal_cadence_steps)\\n        self.dock_position_tol = float(dock_position_tol_m)\\n        self.dock_velocity_tol = float(dock_velocity_tol_mps)\\n        self.dock_attitude_tol_deg = float(dock_attitude_tol_deg)\\n        self.dock_omega_tol = float(dock_omega_tol_radps)\\n        self.diverged_bound = float(diverged_bound_m)\\n        self.max_omega_radps = float(max_omega_radps)\\n        self.reward_weights = dict(reward_weights or DEFAULT_REWARD_WEIGHTS)\\n        self.success_reward = float(success_reward)\\n        self.diverged_penalty = float(diverged_penalty)\\n        self.fuel_out_penalty = float(fuel_out_penalty)\\n        self.shaping_scale = float(shaping_scale)\\n        self.fuel_rate = float(fuel_consumption_rate_pct_per_s)\\n        self.initial_range = tuple(initial_range_m)\\n        self.motion_limits = motion_limits\\n\\n        self.observation_space = spaces.Box(\\n            low=-np.inf, high=np.inf, shape=(OBS_DIM,), dtype=np.float32\\n        )\\n        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(ACTION_DIM,), dtype=np.float32)\\n\\n        self._rng = np.random.default_rng(seed)\\n        self._planner = HeuristicAgent()\\n        self._state: RigidBodyState | None = None\\n        self._subgoal: LlmSubgoal | None = None\\n        self._step = 0\\n        self._prev_range_err = 0.0\\n\\n    # ------------------------------------------------------------------\\n    # Gymnasium API\\n    # ------------------------------------------------------------------\\n\\n    def reset(self, *, seed: int | None = None, options: dict | None = None):\\n        super().reset(seed=seed)\\n        if seed is not None:\\n            self._rng = np.random.default_rng(seed)\\n\\n        range_m = float(self._rng.uniform(*self.initial_range))\\n        direction = self._rng.normal(size=3)\\n        direction /= max(float(np.linalg.norm(direction)), 1e-12)\\n        pos = direction * range_m\\n        vel = self._rng.normal(0.0, 0.5, 3)\\n        if self.motion_limits is not None:\\n            vel = limit_velocity(vel, self.motion_limits)\\n        quat = self._random_unit_quat()\\n        omega = self._rng.normal(0.0, 0.02, 3)\\n\\n        self._state = RigidBodyState(\\n            position=pos,\\n            velocity=vel,\\n            quat=quat,\\n            omega=omega,\\n            fuel_pct=100.0,\\n            mass_kg=self.mass_kg,\\n            inertia=self.inertia,\\n        )\\n        self._subgoal = self._plan_subgoal()\\n        self._step = 0\\n        self._prev_range_err = self._range_error()\\n        return self._observe(), {}\\n\\n    def step(self, action):\\n        action = np.clip(np.asarray(action, dtype=float), -1.0, 1.0)\\n        accel = action[:3] * self.max_accel\\n        torque = action[3:] * self.max_torque\\n        command = Command(force_n=accel * self.mass_kg, torque_nm=torque)\\n        if self.motion_limits is not None:\\n            command, _ = enforce_command(\\n                self._state, command, self.dt, self.n, np.zeros(3), self.motion_limits\\n            )\\n\\n        self._state = propagate_chaser(self._state, self.dt, self.n, command)\\n        if np.linalg.norm(command.force_n) > 1e-6:\\n            self._state.fuel_pct = max(0.0, self._state.fuel_pct - self.fuel_rate * self.dt)\\n        self._step += 1\\n\\n        # Re-issue the sub-goal on the same cadence the service uses, so the\\n        # policy sees the same moving-target distribution at inference.\\n        if self._step % self.subgoal_cadence_steps == 0:\\n            self._subgoal = self._plan_subgoal()\\n\\n        reward, terminated, reason = self._reward(action)\\n        truncated = (self._step >= self.max_steps) and not terminated\\n        obs = self._observe()\\n        info = {\\n            \\"is_success\\": reason == \\"docked\\",\\n            \\"termination\\": reason,\\n            \\"range_m\\": float(np.linalg.norm(self._state.position)),\\n            \\"subgoal_target_m\\": [float(v) for v in self._subgoal.target_position_m],\\n        }\\n        return obs, float(reward), bool(terminated), bool(truncated), info\\n\\n    # ------------------------------------------------------------------\\n    # Reward / termination\\n    # ------------------------------------------------------------------\\n\\n    def _reward(self, action: np.ndarray) -> tuple[float, bool, str | None]:\\n        s = self._state\\n        w = self.reward_weights\\n        range_err = self._range_error()\\n        shaping = self.shaping_scale * (self._prev_range_err - range_err)\\n        self._prev_range_err = range_err\\n\\n        dv = s.velocity - np.asarray(self._subgoal.target_velocity_mps, dtype=float)\\n        att_err_rad = math.radians(attitude_error_deg(s.quat, _IDENTITY_QUAT))\\n\\n        r = (\\n            -w[\\"w_pos\\"] * min(range_err, 10.0) * 0.02\\n            - w[\\"w_vel\\"] * float(np.linalg.norm(dv))\\n            - w[\\"w_att\\"] * att_err_rad\\n            - w[\\"w_ctrl\\"] * float(np.linalg.norm(action))\\n            + shaping\\n        )\\n\\n        if self._is_docked(s):\\n            return r + max(100.0, self.success_reward), True, \\"docked\\"\\n        if (\\n            float(np.linalg.norm(s.position)) > self.diverged_bound\\n            or float(np.linalg.norm(s.omega)) > self.max_omega_radps\\n        ):\\n            return r - self.diverged_penalty, True, \\"diverged\\"\\n        if s.fuel_pct <= 0.0:\\n            return r - self.fuel_out_penalty, True, \\"fuel_out\\"\\n        return r, False, None\\n\\n    def _is_docked(self, s: RigidBodyState) -> bool:\\n        return capture_checks(\\n            s,\\n            self.dock_position_tol,\\n            self.dock_velocity_tol,\\n            self.dock_attitude_tol_deg,\\n            self.dock_omega_tol,\\n        )[\\"eligible\\"]\\n\\n    # ------------------------------------------------------------------\\n    # Internals\\n    # ------------------------------------------------------------------\\n\\n    def _observe(self) -> np.ndarray:\\n        return build_rl_observation(self._state, self._subgoal)\\n\\n    def _range_error(self) -> float:\\n        \\"\\"\\"‖state - subgoal_target‖, the primary tracking error.\\"\\"\\"\\n        return float(\\n            np.linalg.norm(\\n                np.asarray(self._state.position, dtype=float)\\n                - np.asarray(self._subgoal.target_position_m, dtype=float)\\n            )\\n        )\\n\\n    def _plan_subgoal(self) -> LlmSubgoal:\\n        s = self._state\\n        inp = ReasoningInput(\\n            step=self._step,\\n            sim_time_s=self._step * self.dt,\\n            phase=classify_phase(float(np.linalg.norm(s.position))),\\n            range_m=float(np.linalg.norm(s.position)),\\n            position_m=[float(v) for v in s.position],\\n            velocity_mps=[float(v) for v in s.velocity],\\n            orientation_quat=[float(v) for v in s.quat],\\n            angular_velocity_radps=[float(v) for v in s.omega],\\n            fuel_remaining_pct=s.fuel_pct,\\n            controller=\\"rl\\",\\n            anomaly_context=[],\\n            priority_change=None,\\n        )\\n        return self._planner.plan(inp)\\n\\n    def _random_unit_quat(self) -> np.ndarray:\\n        q = self._rng.normal(size=4)\\n        norm = float(np.linalg.norm(q))\\n        q = q / norm if norm > 0.0 else np.array([1.0, 0.0, 0.0, 0.0])\\n        if q[0] < 0.0:  # convention: scalar part >= 0\\n            q = -q\\n        return q\\n\\n    # ------------------------------------------------------------------\\n    # Expert-hook (used by the BC warm-start in train_ppo.py)\\n    # ------------------------------------------------------------------\\n\\n    @property\\n    def current_state(self) -> RigidBodyState:\\n        return self._state\\n\\n    @property\\n    def current_subgoal(self) -> LlmSubgoal:\\n        return self._subgoal\\n\\n    def command_to_action(self, command: Command) -> np.ndarray:\\n        \\"\\"\\"Convert a controller Command (N, N·m) to the env\'s [-1,1] action.\\"\\"\\"\\n        accel = np.asarray(command.force_n, dtype=float) / self.mass_kg\\n        torque = np.asarray(command.torque_nm, dtype=float)\\n        action = np.concatenate([accel / self.max_accel, torque / self.max_torque])\\n        return np.clip(action, -1.0, 1.0).astype(np.float32)\\n", "src/app/simulation/rl/evaluate_policy.py": "\\"\\"\\"Evaluate controllers through the REAL SimulationRunner.\\n\\nRolls out N seeds × controller types and reports the outcomes that matter —\\ndocked / diverged / fuel-out / timed-out (and diverged) counts, plus mean\\nΔv / ISE / LLM-call metrics — rather than training reward.\\n\\nWhy this exists: a rising training ``ep_rew_mean`` is necessary but NOT\\nsufficient evidence the policy learned the intended behavior. A policy that\\nlearns to fail fast and cheap (fly away immediately) produces a clean-looking\\nimproving reward curve while doing the opposite of the task. This script runs\\nthe actual production pipeline (same runner, same dynamics, same sub-goal\\nplanner) and reports the terminal outcomes — run it periodically against\\ncheckpoints while training.\\n\\nExample\\n-------\\n    # All classical controllers, no RL model needed:\\n    python -m app.simulation.rl.evaluate_policy --controllers pid,lqr --seeds 5\\n\\n    # Include the trained RL policy:\\n    python -m app.simulation.rl.evaluate_policy --controllers pid,lqr,rl --seeds 5 \\\\\\n        --rl-model models/rl/ppo_dock.zip --rl-normalize models/rl/ppo_dock_vecnormalize.pkl\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport asyncio\\nimport json\\nimport os\\nimport sys\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nfrom app.llm.reasoning_agent import HeuristicAgent\\nfrom app.schemas.docking import (\\n    OrbitalElements,\\n    SatelliteState,\\n    SimulateRequest,\\n    SimulationConfig,\\n)\\nfrom app.sessions.store import Session\\nfrom app.simulation.runner import SimulationRunner\\n\\nLEO_SMA_M = 6_878_000.0\\n\\n\\ndef make_request(\\n    controller: str, seed: int, range_m: float, max_steps: int, max_range_m: float\\n) -> SimulateRequest:\\n    orbit = OrbitalElements(semi_major_axis_m=LEO_SMA_M)\\n    primary = SatelliteState(orbital_elements=orbit)\\n    secondary = SatelliteState(position_m=[range_m, 0.0, 0.0], orbital_elements=orbit)\\n    return SimulateRequest(\\n        primary=primary,\\n        secondary=secondary,\\n        config=SimulationConfig(\\n            controller=controller,\\n            max_steps=max_steps,\\n            max_range_m=max_range_m,\\n            llm_enabled=True,\\n            llm_cadence_steps=10,\\n            seed=seed,\\n        ),\\n    )\\n\\n\\nasync def run_trial(controller: str, seed: int, args: argparse.Namespace) -> dict[str, Any]:\\n    request = make_request(controller, seed, args.range_m, args.max_steps, args.max_range_m)\\n    session = Session(session_id=f\\"{controller}-seed{seed}\\")\\n    runner = SimulationRunner(session=session, request=request, agent=HeuristicAgent())\\n    await runner.run()\\n    summary = session.summary\\n    assert summary is not None\\n\\n    final = summary.final_secondary\\n    final_range = (\\n        float(sum(summary.final_secondary.position_m[i] ** 2 for i in range(3)) ** 0.5)\\n        if final\\n        else -1.0\\n    )\\n    m = summary.metrics\\n    return {\\n        \\"controller\\": controller,\\n        \\"seed\\": seed,\\n        \\"status\\": summary.status,\\n        \\"stop_reason\\": summary.stop_reason.value if summary.stop_reason else None,\\n        \\"failure_reason\\": summary.failure_reason.value if summary.failure_reason else None,\\n        \\"steps\\": summary.steps_completed,\\n        \\"final_range_m\\": round(final_range, 3),\\n        \\"delta_v_mps\\": round(m.delta_v_mps, 3),\\n        \\"ise\\": round(m.ise, 3),\\n        \\"llm_calls\\": m.total_llm_calls,\\n        \\"llm_fallbacks\\": m.llm_fallback_calls,\\n    }\\n\\n\\ndef _parse_controllers(raw: str) -> list[str]:\\n    controllers = [c.strip() for c in raw.split(\\",\\") if c.strip()]\\n    allowed = {\\"pid\\", \\"lqr\\", \\"rl\\"}\\n    bad = set(controllers) - allowed\\n    if bad:\\n        sys.exit(f\\"Unknown controller(s): {sorted(bad)} (allowed: {sorted(allowed)})\\")\\n    return controllers\\n\\n\\ndef parse_args() -> argparse.Namespace:\\n    p = argparse.ArgumentParser(\\n        description=\\"Evaluate controllers through the real SimulationRunner.\\"\\n    )\\n    p.add_argument(\\"--controllers\\", type=str, default=\\"pid,lqr\\", help=\\"Comma-separated: pid,lqr,rl\\")\\n    p.add_argument(\\"--seeds\\", type=int, default=5, help=\\"Number of random seeds per controller.\\")\\n    p.add_argument(\\"--range-m\\", type=float, default=50.0, help=\\"Initial chaser range [m].\\")\\n    p.add_argument(\\"--max-steps\\", type=int, default=600)\\n    p.add_argument(\\n        \\"--max-range-m\\",\\n        type=float,\\n        default=10_000.0,\\n        help=\\"Divergence guard bound (None disables).\\",\\n    )\\n    p.add_argument(\\n        \\"--rl-model\\", type=str, default=None, help=\\"Trained policy zip (controller \'rl\').\\"\\n    )\\n    p.add_argument(\\n        \\"--rl-normalize\\", type=str, default=None, help=\\"VecNormalize stats pkl (controller \'rl\').\\"\\n    )\\n    p.add_argument(\\"--out\\", type=str, default=None, help=\\"Optional JSON report path.\\")\\n    return p.parse_args()\\n\\n\\ndef main() -> None:\\n    args = parse_args()\\n    controllers = _parse_controllers(args.controllers)\\n\\n    if \\"rl\\" in controllers:\\n        model = args.rl_model or os.environ.get(\\"LLM_DOCKING_RL_MODEL_PATH\\")\\n        norm = args.rl_normalize or os.environ.get(\\"LLM_DOCKING_RL_NORMALIZE_PATH\\")\\n        if not model or not norm:\\n            sys.exit(\\n                \\"controller \'rl\' needs a trained policy: pass --rl-model/--rl-normalize \\"\\n                \\"(or set LLM_DOCKING_RL_MODEL_PATH / LLM_DOCKING_RL_NORMALIZE_PATH). \\"\\n                \\"Train one with: python -m app.simulation.rl.train_ppo\\"\\n            )\\n        for path in (model, norm):\\n            if not Path(path).is_file():\\n                sys.exit(f\\"RL artifact not found: {path}\\")\\n        # Must be set before the first Settings() is cached (SimulationRunner).\\n        os.environ[\\"LLM_DOCKING_RL_MODEL_PATH\\"] = model\\n        os.environ[\\"LLM_DOCKING_RL_NORMALIZE_PATH\\"] = norm\\n\\n    async def _run_all() -> list[dict[str, Any]]:\\n        rows: list[dict[str, Any]] = []\\n        for controller in controllers:\\n            for seed in range(args.seeds):\\n                row = await run_trial(controller, seed, args)\\n                rows.append(row)\\n                print(\\n                    f\\"  {controller:<4} seed {seed}: {row[\'status\']:<7} \\"\\n                    f\\"steps={row[\'steps\']:<4} range={row[\'final_range_m\']:>8.2f}m \\"\\n                    f\\"dv={row[\'delta_v_mps\']:>7.2f} m/s ise={row[\'ise\']:>10.2f}\\"\\n                )\\n        return rows\\n\\n    rows = asyncio.run(_run_all())\\n\\n    # ---- aggregate table ----\\n    print(\\"\\\\n=== Summary (docked / diverged / fuel-out / timed-out) ===\\")\\n    header = f\\"{\'controller\':<10} {\'docked\':>7} {\'diverged\':>8} {\'fuel_out\':>8} {\'timed_out\':>9} {\'mean dv\':>9} {\'mean ise\':>10}\\"\\n    print(header)\\n    print(\\"-\\" * len(header))\\n    report: dict[str, Any] = {\\"trials\\": rows, \\"aggregate\\": {}}\\n    for controller in controllers:\\n        sub = [r for r in rows if r[\\"controller\\"] == controller]\\n        counts = {\\n            \\"docked\\": sum(1 for r in sub if r[\\"status\\"] == \\"docked\\"),\\n            \\"diverged\\": sum(1 for r in sub if r[\\"failure_reason\\"] == \\"diverged\\"),\\n            \\"fuel_out\\": sum(1 for r in sub if r[\\"failure_reason\\"] == \\"fuel_exhausted\\"),\\n            \\"timed_out\\": sum(\\n                1 for r in sub if r[\\"failure_reason\\"] in (\\"max_steps_exceeded\\", \\"max_time_exceeded\\")\\n            ),\\n        }\\n        docked = [r for r in sub if r[\\"status\\"] == \\"docked\\"] or sub\\n        agg = {\\n            **counts,\\n            \\"mean_delta_v_mps\\": round(sum(r[\\"delta_v_mps\\"] for r in docked) / len(docked), 3),\\n            \\"mean_ise\\": round(sum(r[\\"ise\\"] for r in docked) / len(docked), 3),\\n            \\"mean_llm_calls\\": round(sum(r[\\"llm_calls\\"] for r in sub) / len(sub), 2),\\n        }\\n        report[\\"aggregate\\"][controller] = agg\\n        print(\\n            f\\"{controller:<10} {agg[\'docked\']:>7} {agg[\'diverged\']:>8} {agg[\'fuel_out\']:>8} \\"\\n            f\\"{agg[\'timed_out\']:>9} {agg[\'mean_delta_v_mps\']:>9} {agg[\'mean_ise\']:>10}\\"\\n        )\\n\\n    if args.out:\\n        Path(args.out).write_text(json.dumps(report, indent=2))\\n        print(f\\"\\\\nReport written to {args.out}\\")\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "src/app/simulation/rl/proofs.py": "\\"\\"\\"Export real training curves, randomized controller evaluations, and SHA-256 hashes.\\"\\"\\"\\n\\nimport argparse\\nimport asyncio\\nimport csv\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\n\\nimport matplotlib\\n\\nmatplotlib.use(\\"Agg\\")\\nimport matplotlib.pyplot as plt\\nimport numpy as np\\nfrom tensorboard.backend.event_processing.event_accumulator import EventAccumulator\\n\\nfrom app.config import get_settings\\nfrom app.llm.reasoning_agent import HeuristicAgent\\nfrom app.schemas.docking import OrbitalElements, SatelliteState, SimulateRequest, SimulationConfig\\nfrom app.sessions.store import Session\\nfrom app.simulation.runner import SimulationRunner\\n\\n\\nasync def evaluate(directory, episodes, start_seed, checkpoint):\\n    import os\\n\\n    os.environ[\\"LLM_DOCKING_RL_MODEL_PATH\\"] = str(directory / f\\"{checkpoint}.zip\\")\\n    os.environ[\\"LLM_DOCKING_RL_NORMALIZE_PATH\\"] = str(directory / \\"ppo_dock_vecnormalize.pkl\\")\\n    get_settings.cache_clear()\\n    rows = []\\n    for controller, advisor in [(\\"lqr\\", False), (\\"pid\\", False), (\\"rl\\", False), (\\"lqr\\", True)]:\\n        for seed in range(start_seed, start_seed + episodes):\\n            rng = np.random.default_rng(seed)\\n            direction = rng.normal(size=3)\\n            position = direction / np.linalg.norm(direction) * rng.uniform(5, 80)\\n            quat = rng.normal(size=4)\\n            quat /= np.linalg.norm(quat)\\n            request = SimulateRequest(\\n                primary=SatelliteState(orbital_elements=OrbitalElements(semi_major_axis_m=6878000)),\\n                secondary=SatelliteState(\\n                    position_m=position.tolist(),\\n                    velocity_mps=rng.normal(0, 0.15, 3).tolist(),\\n                    orientation_quat=quat.tolist(),\\n                    angular_velocity_radps=rng.normal(0, 0.01, 3).tolist(),\\n                ),\\n                config=SimulationConfig(\\n                    controller=controller, rl_advisor_enabled=advisor, max_steps=600, seed=seed\\n                ),\\n            )\\n            session = Session(f\\"{controller}-{\'hybrid\' if advisor else \'baseline\'}-{seed}\\")\\n            await SimulationRunner(session, request, agent=HeuristicAgent()).run()\\n            summary = session.summary\\n            final = summary.final_secondary\\n            rows.append(\\n                {\\n                    \\"controller\\": controller,\\n                    \\"rl_advisor\\": advisor,\\n                    \\"seed\\": seed,\\n                    \\"status\\": summary.status,\\n                    \\"failure_reason\\": summary.failure_reason,\\n                    \\"initial_position_m\\": position.tolist(),\\n                    \\"steps\\": summary.steps_completed,\\n                    \\"final_range_m\\": float(np.linalg.norm(final.position_m)) if final else None,\\n                    \\"final_speed_mps\\": float(np.linalg.norm(final.velocity_mps)) if final else None,\\n                    \\"delta_v_mps\\": summary.metrics.delta_v_mps,\\n                    \\"capture\\": session.capture,\\n                }\\n            )\\n            print(f\\"{controller} advisor={advisor} seed={seed}: {summary.status}\\", flush=True)\\n    return rows\\n\\n\\ndef export_curves(directory):\\n    rows = []\\n    for file in sorted((directory / \\"tensorboard\\").rglob(\\"events.out.tfevents.*\\")):\\n        accumulator = EventAccumulator(str(file), size_guidance={\\"scalars\\": 0})\\n        accumulator.Reload()\\n        for tag in accumulator.Tags()[\\"scalars\\"]:\\n            rows += [\\n                {\\"tag\\": tag, \\"step\\": x.step, \\"value\\": x.value, \\"wall_time\\": x.wall_time}\\n                for x in accumulator.Scalars(tag)\\n            ]\\n    with (directory / \\"tensorboard_scalars.csv\\").open(\\"w\\", newline=\\"\\") as file:\\n        writer = csv.DictWriter(file, fieldnames=[\\"tag\\", \\"step\\", \\"value\\", \\"wall_time\\"])\\n        writer.writeheader()\\n        writer.writerows(rows)\\n    tags = [\\n        tag\\n        for tag in [\\n            \\"rollout/ep_rew_mean\\",\\n            \\"train/value_loss\\",\\n            \\"train/policy_gradient_loss\\",\\n            \\"eval/success_rate\\",\\n        ]\\n        if any(row[\\"tag\\"] == tag for row in rows)\\n    ]\\n    if tags:\\n        fig, axes = plt.subplots(len(tags), 1, figsize=(9, 3 * len(tags)), squeeze=False)\\n        for axis, tag in zip(axes.flat, tags, strict=True):\\n            data = sorted([r for r in rows if r[\\"tag\\"] == tag], key=lambda r: r[\\"step\\"])\\n            axis.plot([r[\\"step\\"] for r in data], [r[\\"value\\"] for r in data])\\n            axis.set(title=tag, xlabel=\\"Environment timesteps\\")\\n            axis.grid(alpha=0.2)\\n        fig.tight_layout()\\n        fig.savefig(directory / \\"training_curves.png\\", dpi=160)\\n        plt.close(fig)\\n    return len(rows)\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--run-dir\\", type=Path, required=True)\\n    parser.add_argument(\\"--episodes\\", type=int, default=30)\\n    parser.add_argument(\\"--start-seed\\", type=int, default=50000)\\n    parser.add_argument(\\"--checkpoint\\", default=\\"ppo_dock\\")\\n    args = parser.parse_args()\\n    if args.episodes < 1:\\n        parser.error(\\"At least one held-out episode is required\\")\\n    directory = args.run_dir\\n    rows = asyncio.run(evaluate(directory, args.episodes, args.start_seed, args.checkpoint))\\n    aggregates = []\\n    for controller, advisor in [(\\"lqr\\", False), (\\"pid\\", False), (\\"rl\\", False), (\\"lqr\\", True)]:\\n        trials = [r for r in rows if r[\\"controller\\"] == controller and r[\\"rl_advisor\\"] == advisor]\\n        count = sum(r[\\"status\\"] == \\"docked\\" for r in trials)\\n        n = len(trials)\\n        p = count / n\\n        # Wilson interval: small samples must carry uncertainty.\\n        center = (p + 1.96**2 / (2 * n)) / (1 + 1.96**2 / n)\\n        margin = 1.96 * np.sqrt(p * (1 - p) / n + 1.96**2 / (4 * n**2)) / (1 + 1.96**2 / n)\\n        aggregates.append(\\n            {\\n                \\"controller\\": controller,\\n                \\"rl_advisor\\": advisor,\\n                \\"docked\\": count,\\n                \\"trials\\": n,\\n                \\"success_rate\\": p,\\n                \\"wilson_95_percent\\": [max(0.0, center - margin), min(1.0, center + margin)],\\n            }\\n        )\\n    report = {\\n        \\"scope\\": \\"Research simulation; randomized 5-80 m starts, random attitude, 600 s budget; heuristic planner, no provider calls\\",\\n        \\"checkpoint\\": args.checkpoint,\\n        \\"aggregate\\": aggregates,\\n        \\"trials\\": rows,\\n    }\\n    (directory / \\"evaluation.json\\").write_text(json.dumps(report, indent=2))\\n    scalars = export_curves(directory)\\n    files = [p for p in directory.rglob(\\"*\\") if p.is_file() and p.name != \\"manifest.json\\"]\\n    manifest = {\\n        \\"files\\": {\\n            str(p.relative_to(directory)): hashlib.sha256(p.read_bytes()).hexdigest() for p in files\\n        },\\n        \\"tensorboard_scalar_count\\": scalars,\\n        \\"source_sha256\\": {\\n            str(p.relative_to(Path(\\"src\\"))): hashlib.sha256(p.read_bytes()).hexdigest()\\n            for p in Path(\\"src\\").rglob(\\"*.py\\")\\n        },\\n    }\\n    (directory / \\"manifest.json\\").write_text(json.dumps(manifest, indent=2))\\n    print(json.dumps(aggregates, indent=2))\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "src/app/simulation/rl/train_ppo.py": "\\"\\"\\"Train an expert-initialized PPO policy; save the data needed to reproduce it.\\n\\nUse --device cuda:0 in Kaggle. Two independent processes can use cuda:0 and\\ncuda:1 with different seeds. A small MLP does not pool the GPUs\' memory.\\n\\"\\"\\"\\n\\nimport argparse\\nimport hashlib\\nimport importlib.metadata\\nimport json\\nimport platform\\nimport time\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\nfrom stable_baselines3 import PPO\\nfrom stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback\\nfrom stable_baselines3.common.monitor import Monitor\\nfrom stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize\\nfrom torch.utils.data import DataLoader, TensorDataset\\n\\nfrom app.schemas.docking import MotionLimits\\nfrom app.simulation.controllers import ControlContext, build_controller\\nfrom app.simulation.rl.docking_env import DockingEnv\\n\\n\\ndef environment(seed, max_steps=600):\\n    return DockingEnv(\\n        seed=seed,\\n        max_steps=max_steps,\\n        initial_range_m=(5.0, 80.0),\\n        motion_limits=MotionLimits(),\\n        diverged_penalty=200.0,\\n        shaping_scale=3.0,\\n    )\\n\\n\\ndef expert_trajectories(episodes, seed, max_steps):\\n    env = environment(seed, max_steps)\\n    controller = build_controller(\\"lqr\\", env.mass_kg, env.inertia, env.n)\\n    observations, actions, outcomes = [], [], []\\n    for episode in range(episodes):\\n        obs, _ = env.reset(seed=seed + episode)\\n        ctx = ControlContext()\\n        for step in range(max_steps):\\n            if step % env.subgoal_cadence_steps == 0:\\n                controller.reset(ctx)\\n            command = controller.compute(env.current_state, env.current_subgoal, env.dt, ctx)\\n            action = env.command_to_action(command)\\n            observations.append(obs)\\n            actions.append(action)\\n            obs, _, terminated, truncated, info = env.step(action)\\n            if terminated or truncated:\\n                outcomes.append(info[\\"termination\\"] or \\"timeout\\")\\n                break\\n    env.close()\\n    return (\\n        np.asarray(observations, dtype=np.float32),\\n        np.asarray(actions, dtype=np.float32),\\n        outcomes,\\n    )\\n\\n\\ndef warm_start(model, normalizer, observations, actions, epochs, device):\\n    # Fit observation statistics BEFORE cloning; PPO sees these exact normalized inputs.\\n    normalizer.obs_rms.update(observations)\\n    normalizer.training = False\\n    normalized = normalizer.normalize_obs(observations)\\n    loader = DataLoader(\\n        TensorDataset(torch.from_numpy(normalized), torch.from_numpy(actions)),\\n        batch_size=512,\\n        shuffle=True,\\n    )\\n    optimizer = torch.optim.Adam(model.policy.parameters(), lr=0.001)\\n    losses = []\\n    for _ in range(epochs):\\n        total = 0.0\\n        for obs, action in loader:\\n            prediction = model.policy.get_distribution(obs.to(device)).distribution.mean\\n            loss = torch.nn.functional.mse_loss(prediction, action.to(device))\\n            optimizer.zero_grad()\\n            loss.backward()\\n            optimizer.step()\\n            total += loss.item() * len(obs)\\n        losses.append(total / len(observations))\\n    with torch.no_grad():\\n        model.policy.log_std.fill_(-3.0)\\n    return losses\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--timesteps\\", type=int, default=200000)\\n    parser.add_argument(\\"--out-dir\\", type=Path, default=Path(\\"models/rl\\"))\\n    parser.add_argument(\\"--device\\", default=\\"cpu\\")\\n    parser.add_argument(\\"--seed\\", type=int, default=42)\\n    parser.add_argument(\\"--n-seeds\\", type=int, default=8)\\n    parser.add_argument(\\"--n-steps\\", type=int, default=256)\\n    parser.add_argument(\\"--batch-size\\", type=int, default=512)\\n    parser.add_argument(\\"--lr\\", type=float, default=0.00001)\\n    parser.add_argument(\\"--bc-episodes\\", type=int, default=80)\\n    parser.add_argument(\\"--bc-epochs\\", type=int, default=30)\\n    parser.add_argument(\\"--max-steps\\", type=int, default=600)\\n    parser.add_argument(\\"--eval-freq\\", type=int, default=20000)\\n    parser.add_argument(\\"--eval-episodes\\", type=int, default=12)\\n    parser.add_argument(\\"--resume\\", type=Path)\\n    parser.add_argument(\\"--resume-normalize\\", type=Path)\\n    args = parser.parse_args()\\n    if args.timesteps < 1 or args.n_seeds < 1 or args.bc_episodes < 1:\\n        parser.error(\\"Timesteps, environment count, and expert episodes must be positive\\")\\n    if args.device.startswith(\\"cuda\\") and not torch.cuda.is_available():\\n        parser.error(\\"CUDA requested but no CUDA runtime is available\\")\\n    if args.resume and not args.resume_normalize:\\n        parser.error(\\"Resume requires the matching --resume-normalize checkpoint\\")\\n    torch.set_num_threads(2)\\n    args.out_dir.mkdir(parents=True, exist_ok=True)\\n    metadata = {k: str(v) if isinstance(v, Path) else v for k, v in vars(args).items()}\\n    metadata.update(\\n        python=platform.python_version(),\\n        torch=torch.__version__,\\n        capture_range_m=1.25,\\n        initial_range_m=[5, 80],\\n        dt_s=1.0,\\n        observation=\\"14 values: subgoal position/velocity error, quaternion, rotation rate, fuel\\",\\n        reward=\\"bounded position cost, velocity and attitude cost, action cost, range progress, capture bonus 100\\",\\n        gpu=[\\n            {\\n                \\"name\\": torch.cuda.get_device_properties(i).name,\\n                \\"memory_bytes\\": torch.cuda.get_device_properties(i).total_memory,\\n            }\\n            for i in range(torch.cuda.device_count())\\n        ],\\n    )\\n    metadata[\\"dependencies\\"] = {\\n        name: importlib.metadata.version(name)\\n        for name in [\\"stable-baselines3\\", \\"gymnasium\\", \\"torch\\", \\"numpy\\", \\"tensorboard\\", \\"pydantic\\"]\\n    }\\n    metadata[\\"source_sha256\\"] = {\\n        str(path.relative_to(Path(\\"src\\"))): hashlib.sha256(path.read_bytes()).hexdigest()\\n        for path in Path(\\"src\\").rglob(\\"*.py\\")\\n    }\\n    metadata[\\"motion_limits\\"] = MotionLimits().model_dump()\\n    (args.out_dir / \\"parameters.json\\").write_text(json.dumps(metadata, indent=2))\\n    env = DummyVecEnv(\\n        [\\n            lambda i=i: Monitor(environment(args.seed + i, args.max_steps))\\n            for i in range(args.n_seeds)\\n        ]\\n    )\\n    normalizer = VecNormalize(env, norm_obs=True, norm_reward=False)\\n    if args.resume:\\n        normalizer = VecNormalize.load(str(args.resume_normalize), env)\\n        normalizer.training = False\\n        normalizer.norm_reward = False\\n        model = PPO.load(str(args.resume), env=normalizer, device=args.device)\\n        model.tensorboard_log = str(args.out_dir / \\"tensorboard\\")\\n    else:\\n        model = PPO(\\n            \\"MlpPolicy\\",\\n            normalizer,\\n            device=args.device,\\n            seed=args.seed,\\n            policy_kwargs={\\"net_arch\\": [128, 128]},\\n            n_steps=args.n_steps,\\n            batch_size=args.batch_size,\\n            learning_rate=args.lr,\\n            gamma=0.995,\\n            ent_coef=0.0,\\n            clip_range=0.1,\\n            n_epochs=4,\\n            verbose=0,\\n            tensorboard_log=str(args.out_dir / \\"tensorboard\\"),\\n        )\\n        observations, actions, outcomes = expert_trajectories(\\n            args.bc_episodes, args.seed, args.max_steps\\n        )\\n        losses = warm_start(model, normalizer, observations, actions, args.bc_epochs, args.device)\\n        (args.out_dir / \\"behavior_cloning.json\\").write_text(\\n            json.dumps(\\n                {\\"losses\\": losses, \\"outcomes\\": outcomes, \\"samples\\": len(observations)}, indent=2\\n            )\\n        )\\n        model.save(args.out_dir / \\"expert_initialized\\")\\n        normalizer.save(str(args.out_dir / \\"expert_initialized_vecnormalize.pkl\\"))\\n    eval_env = VecNormalize(\\n        DummyVecEnv([lambda: Monitor(environment(args.seed + 10000, args.max_steps))]),\\n        training=False,\\n        norm_reward=False,\\n    )\\n    eval_env.obs_rms = normalizer.obs_rms\\n    evaluation = EvalCallback(\\n        eval_env,\\n        best_model_save_path=str(args.out_dir),\\n        log_path=str(args.out_dir),\\n        n_eval_episodes=args.eval_episodes,\\n        eval_freq=max(1, args.eval_freq // args.n_seeds),\\n        deterministic=True,\\n        verbose=0,\\n    )\\n    checkpoint = CheckpointCallback(\\n        save_freq=max(1, args.eval_freq // args.n_seeds),\\n        save_path=str(args.out_dir / \\"checkpoints\\"),\\n        name_prefix=\\"ppo_dock\\",\\n        save_vecnormalize=True,\\n    )\\n    start = time.perf_counter()\\n    model.learn(\\n        total_timesteps=args.timesteps,\\n        callback=[evaluation, checkpoint],\\n        reset_num_timesteps=not bool(args.resume),\\n    )\\n    model.save(args.out_dir / \\"ppo_dock\\")\\n    normalizer.save(str(args.out_dir / \\"ppo_dock_vecnormalize.pkl\\"))\\n    metadata.update(\\n        actual_timesteps=model.num_timesteps, elapsed_seconds=time.perf_counter() - start\\n    )\\n    (args.out_dir / \\"parameters.json\\").write_text(json.dumps(metadata, indent=2))\\n    eval_env.close()\\n    normalizer.close()\\n    print(json.dumps({\\"completed\\": model.num_timesteps, \\"out_dir\\": str(args.out_dir)}))\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "src/app/simulation/rl/__init__.py": "\\"\\"\\"RL/PPO docking: Gymnasium env, training and evaluation CLIs.\\n\\n* ``docking_env.DockingEnv`` — the env (same physics as the service runner)\\n* ``train_ppo.py`` — BC warm-start + PPO training (``python -m ...train_ppo``)\\n* ``evaluate_policy.py`` — honest outcome-based eval through SimulationRunner\\n\\"\\"\\"\\n\\nfrom app.simulation.rl.docking_env import DockingEnv, build_rl_observation\\n\\n__all__ = [\\"DockingEnv\\", \\"build_rl_observation\\"]\\n", "pyproject.toml": "[build-system]\\nrequires = [\\"setuptools>=68\\"]\\nbuild-backend = \\"setuptools.build_meta\\"\\n\\n[project]\\nname = \\"llm-docking\\"\\nversion = \\"0.1.0\\"\\ndescription = \\"Backend for an LLM-augmented reasoning agent for autonomous satellite rendezvous and docking simulation.\\"\\nreadme = \\"README.md\\"\\nrequires-python = \\">=3.11\\"\\nlicense = { text = \\"MIT\\" }\\ndependencies = [\\n    \\"fastapi>=0.110,<1.0\\",\\n    \\"uvicorn[standard]>=0.29,<1.0\\",\\n    \\"pydantic>=2.6,<3.0\\",\\n    \\"pydantic-settings>=2.2,<3.0\\",\\n    \\"numpy>=1.26,<3.0\\",\\n    \\"scipy>=1.12,<2.0\\",\\n    \\"httpx>=0.27,<1.0\\",\\n    \\"sse-starlette>=2.1,<3.0\\",\\n    \\"structlog>=24.1,<25.0\\",\\n    \\"langchain-core>=1.0,<2.0\\",\\n    \\"langchain-openai>=1.0,<2.0\\",\\n    \\"langchain-anthropic>=1.0,<2.0\\",\\n]\\n\\n[project.optional-dependencies]\\ndev = [\\n    \\"pytest>=8.0,<9.0\\",\\n    \\"pytest-asyncio>=0.23,<1.0\\",\\n    \\"ruff>=0.4,<1.0\\",\\n]\\n# RL/PPO controller (controller=\\"rl\\"): training + inference.\\n# Torch CPU wheels are fine — no GPU is assumed.\\nrl = [\\n    \\"gymnasium>=1.0,<2.0\\",\\n    \\"stable-baselines3>=2.7,<3.0\\",\\n    \\"torch>=2.1\\",\\n    \\"tensorboard>=2.18,<3.0\\",\\n    \\"matplotlib>=3.8,<4.0\\",\\n]\\n\\n[tool.setuptools.packages.find]\\nwhere = [\\"src\\"]\\n\\n[tool.pytest.ini_options]\\nasyncio_mode = \\"auto\\"\\ntestpaths = [\\"tests\\"]\\naddopts = \\"-q\\"\\npythonpath = [\\"src\\"]\\n\\n[tool.ruff]\\nline-length = 100\\ntarget-version = \\"py311\\"\\n\\n[tool.ruff.lint]\\nselect = [\\"E\\", \\"F\\", \\"I\\", \\"UP\\", \\"B\\"]\\nignore = [\\"E501\\"]\\n\\n[tool.ruff.lint.per-file-ignores]\\n\\"tests/test_rl.py\\" = [\\"E402\\"]\\n", "README.md": "Docking Lab research simulator. See notebook documentation."}')
ROOT.mkdir(parents=True, exist_ok=True)
for name, content in PROJECT_FILES.items():
    target = ROOT / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding='utf-8')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(ROOT) + '[rl]'], check=True)
os.chdir(ROOT)
print('Project ready:', ROOT)


## Configure reproducible training

The default uses two seeds, eight environments per run, 100 expert episodes, 50 cloning epochs, and 200,000 PPO timesteps per GPU. Increase the budget for further research. Training starts with cloned LQR trajectories using the same frozen normalized observations as PPO inference. The normalized policy and its VecNormalize file must always stay paired.

The reward penalizes tracking error, velocity error, attitude error, and control effort; rewards progress; adds a 100-point capture bonus; and penalizes divergence. Source hashes, versions, limits, and parameters are recorded automatically.


In [ ]:
SEEDS = [42, 137]
TIMESTEPS = 200_000
EXPERT_EPISODES = 100
CLONING_EPOCHS = 50
RUN_LABEL = 'notebook'
RUNS = [ROOT / 'proofs' / f'{RUN_LABEL}-seed-{seed}' for seed in SEEDS]
print('Run directories:', RUNS)


In [ ]:
training_processes = []
for gpu, (seed, directory) in enumerate(zip(SEEDS, RUNS)):
    directory.mkdir(parents=True, exist_ok=True)
    device = f'cuda:{gpu}' if gpu < torch.cuda.device_count() else 'cpu'
    command = [sys.executable, '-m', 'app.simulation.rl.train_ppo',
               '--device', device, '--seed', str(seed), '--timesteps', str(TIMESTEPS),
               '--bc-episodes', str(EXPERT_EPISODES), '--bc-epochs', str(CLONING_EPOCHS),
               '--out-dir', str(directory)]
    log_file = (directory / 'training.log').open('w')
    process = subprocess.Popen(command, cwd=ROOT, stdout=log_file, stderr=subprocess.STDOUT)
    log_file.close()
    training_processes.append(process)
    print('Started', seed, device, 'PID', process.pid)


## Monitor and inspect TensorBoard

The monitor cell is nonblocking. Rerun it until both processes have exited with code 0. An exception in the log is a failed run, not evidence of completion. The next cell starts TensorBoard within the notebook.


In [ ]:
for process, directory in zip(training_processes, RUNS):
    status = process.poll()
    print(directory.name, 'running' if status is None else f'exit code {status}')
    print((directory / 'training.log').read_text()[-1000:])


In [ ]:
%load_ext tensorboard
%tensorboard --logdir /kaggle/working/llm-docking/proofs --port 6006


## Evaluate fresh held-out scenarios

Run after training completes. Each checkpoint is compared with PID, LQR, direct PPO, and LQR with PPO advice in the real SimulationRunner. The 50 scenarios per method use seeds 70000–70049, randomized 5–80 m directions, random attitudes, small initial velocity and rotation, and a 600-second budget. These seeds are separate from training/evaluation seeds. No paid LLM calls are made in this benchmark.

The JSON records each trial, final distance/speed, capture checks, failures, and Wilson 95% intervals. `proofs.py` also exports every TensorBoard scalar, curve plots, and SHA-256 manifests. Never report these simulated results as hardware certification or guaranteed performance.


In [ ]:
assert all(process.poll() == 0 for process in training_processes), 'Training is still running or failed.'
evaluation_processes = []
for directory in RUNS:
    log_file = (directory / 'evaluation.log').open('w')
    process = subprocess.Popen([sys.executable, '-m', 'app.simulation.rl.proofs',
                                '--run-dir', str(directory), '--episodes', '50', '--start-seed', '70000'],
                               cwd=ROOT, stdout=log_file, stderr=subprocess.STDOUT)
    log_file.close()
    evaluation_processes.append(process)
    print('Evaluating', directory.name, 'PID', process.pid)


In [2]:
for process, directory in zip(evaluation_processes, RUNS):
    print(directory.name, 'running' if process.poll() is None else f'exit code {process.poll()}')
    if (directory / 'evaluation.json').exists():
        report = json.loads((directory / 'evaluation.json').read_text())
        print(json.dumps(report['aggregate'], indent=2))


Final recorded runs: seeds 42 and 137; 200704 PPO timesteps each.
Seed 42: [{"controller": "lqr", "rl_advisor": false, "docked": 50, "trials": 50, "success_rate": 1.0, "wilson_95_percent": [0.9286499658256813, 1.0]}, {"controller": "pid", "rl_advisor": false, "docked": 50, "trials": 50, "success_rate": 1.0, "wilson_95_percent": [0.9286499658256813, 1.0]}, {"controller": "rl", "rl_advisor": false, "docked": 50, "trials": 50, "success_rate": 1.0, "wilson_95_percent": [0.9286499658256813, 1.0]}, {"controller": "lqr", "rl_advisor": true, "docked": 50, "trials": 50, "success_rate": 1.0, "wilson_95_percent": [0.9286499658256813, 1.0]}]
Seed 137: [{"controller": "lqr", "rl_advisor": false, "docked": 50, "trials": 50, "success_rate": 1.0, "wilson_95_percent": [0.9286499658256813, 1.0]}, {"controller": "pid", "rl_advisor": false, "docked": 50, "trials": 50, "success_rate": 1.0, "wilson_95_percent": [0.9286499658256813, 1.0]}, {"controller": "rl", "rl_advisor": false, "docked": 50, "trials": 50,

## Inspect plots and verify every artifact

The following verifies each file against its manifest. A hash detects a changed file; it is not an independent endorsement. Keep raw failures, parameters, source snapshots, and logs alongside the plots when presenting the experiment.


In [ ]:
from IPython.display import display, Image
assert all(process.poll() == 0 for process in evaluation_processes), 'Evaluation is still running or failed.'
for directory in RUNS:
    manifest = json.loads((directory / 'manifest.json').read_text())
    for name, expected in manifest['files'].items():
        actual = hashlib.sha256((directory / name).read_bytes()).hexdigest()
        assert actual == expected, f'Hash mismatch: {directory.name}/{name}'
    print(directory.name, 'verified', len(manifest['files']), 'files')
    display(Image(filename=str(directory / 'training_curves.png')))


## Export results and connect the backend

Download the archive from Kaggle's Output panel. Copy a matched `ppo_dock.zip` and `ppo_dock_vecnormalize.pkl` pair into your local project. Set the two paths in `.env`, restart FastAPI, and select “Use learned PPO advice” in the dashboard. Provider API keys stay on the backend. The dashboard exports its actual event log as session evidence.


In [ ]:
import shutil
archive = shutil.make_archive(str(ROOT.parent / 'docking-evidence'), 'zip', ROOT, 'proofs')
print('Download:', archive)
print('LLM_DOCKING_RL_MODEL_PATH=proofs/<chosen-run>/ppo_dock.zip')
print('LLM_DOCKING_RL_NORMALIZE_PATH=proofs/<chosen-run>/ppo_dock_vecnormalize.pkl')


## Resume an interrupted run

Pass a checkpoint and its matching normalization file, not the statistics from a different seed or final checkpoint. Checkpoints are saved together under each run's `checkpoints/` folder.


In [ ]:
# Example only: replace the filenames with an existing matched checkpoint pair.
# subprocess.run([sys.executable, '-m', 'app.simulation.rl.train_ppo',
#                 '--device', 'cuda:0', '--seed', '42', '--timesteps', '200000',
#                 '--out-dir', str(RUNS[0]),
#                 '--resume', str(RUNS[0] / 'checkpoints/ppo_dock_20000_steps.zip'),
#                 '--resume-normalize', str(RUNS[0] / 'checkpoints/ppo_dock_vecnormalize_20000_steps.pkl')], check=True)
